# 10 · NLP with Pretrained Transformers (Hugging Face)

**Part IV — Sequences and language** · Estimated time: ~120 min

In Notebook 09 · Attention and Transformers we built a GPT from scratch and trained it on a few
hundred kilobytes of Shakespeare. That was the right way to *understand* the architecture, but it is
not how language models are used in practice. Practitioners almost never train a Transformer from
nothing: they take a model that somebody else pretrained on billions of tokens and *adapt* it. This
notebook is about that workflow. We start one level below the model, with the question of how raw
text becomes integers (tokenization), continue with how integers become vectors (embeddings), and
then spend most of our time on the Hugging Face ecosystem: loading pretrained models, fine-tuning
them by hand and with the `Trainer`, and fine-tuning them *cheaply* with low-rank adapters (LoRA),
which we derive and implement ourselves before using the `peft` library. We close with a tour of how
today's large language models are trained, including a from-scratch implementation of the DPO loss.

## Learning objectives

- Explain the trade-offs between character, word and subword tokenization, and implement Byte-Pair Encoding from scratch.
- Use Hugging Face tokenizers correctly: special tokens, attention masks, padding, truncation and offsets.
- Derive the skip-gram objective with negative sampling and train word2vec embeddings in PyTorch.
- Load pretrained Transformers with `AutoModel`/`AutoTokenizer`, run fill-mask, text generation and semantic search.
- Fine-tune a pretrained encoder for classification with a hand-written loop and with the `Trainer` API.
- Derive LoRA, implement it as a wrapper around `nn.Linear` (with weight merging) and apply it with the `peft` library.
- Describe the pretraining, supervised fine-tuning and preference-optimization stages of modern LLM training, and implement the DPO loss.
- Reason about sequence length, memory, mixed precision, licensing and data contamination when using pretrained models.

## Prerequisites

- Notebook 05 · The PyTorch Workflow and Best Practices (datasets, training loops, checkpoints).
- Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs (the Tiny Shakespeare corpus, language modelling).
- Notebook 09 · Attention and Transformers (attention, Transformer blocks, generation and sampling).

## Contents

1. Tokenization: from characters to subwords
2. Word embeddings: word2vec from scratch
3. Using pretrained models
4. Fine-tuning for classification, by hand and with `Trainer`
5. Parameter-efficient fine-tuning: LoRA from scratch and with `peft`
6. How large language models are trained today
7. Practical considerations
8. Exercises · Summary · Further reading

In [ ]:
import math, os, random, time, re, json, inspect, gc
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE = DATA_DIR / "hf_cache"          # every Hugging Face download lands here
HF_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

# Optional heavy libraries: imported inside try/except so that the notebook degrades gracefully.
try:
    import transformers
    from transformers import (AutoTokenizer, AutoModel, AutoModelForMaskedLM, AutoModelForCausalLM,
                              AutoModelForSequenceClassification, DataCollatorWithPadding,
                              get_linear_schedule_with_warmup, Trainer, TrainingArguments, pipeline)
    transformers.logging.set_verbosity_error()
    HF_AVAILABLE = True
    print(f"transformers {transformers.__version__}")
except ImportError as e:
    HF_AVAILABLE = False
    AutoTokenizer = AutoModel = AutoModelForMaskedLM = AutoModelForCausalLM = None
    AutoModelForSequenceClassification = DataCollatorWithPadding = None
    get_linear_schedule_with_warmup = Trainer = TrainingArguments = pipeline = None
    print("transformers is not installed -> all Hugging Face cells will be skipped.\n"
          "  Create the course environment from environment.yml to run them. (", e, ")")
try:
    import datasets as hf_datasets
    print(f"datasets {hf_datasets.__version__}")
except ImportError:
    hf_datasets = None
    print("datasets is not installed -> the in-notebook sentiment data will be used (see environment.yml).")
try:
    import peft
    from peft import LoraConfig, get_peft_model, TaskType
    PEFT_AVAILABLE = True
    print(f"peft {peft.__version__}")
except ImportError:
    peft = LoraConfig = get_peft_model = TaskType = None
    PEFT_AVAILABLE = False
    print("peft is not installed -> the peft cells will be skipped (see environment.yml).")


def hf_load(loader, name, **kwargs):
    """Load a Hugging Face tokenizer/model into DATA_DIR/hf_cache; return None (with a note) on failure."""
    if loader is None:
        print(f"[skipped] transformers is not available -> cannot load '{name}'.")
        return None
    t0 = time.time()
    try:
        obj = loader.from_pretrained(name, cache_dir=str(HF_CACHE), **kwargs)
    except Exception as e:  # no network, gated model, corrupted cache, ...
        print(f"[skipped] could not load '{name}': {type(e).__name__}: {str(e)[:120]}")
        print(f"          (the first use needs internet access; files are cached in {HF_CACHE})")
        return None
    print(f"loaded {loader.__name__}('{name}') in {time.time() - t0:.1f}s")
    return obj


def skip(reason):
    """Print a uniform note for cells that cannot run in the current environment."""
    print(f"[skipped] {reason}")


def count_params(model):
    """Return (total, trainable) parameter counts of a module."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

A word on the flags printed above. `HF_AVAILABLE` tells us whether the `transformers` library could be
imported; `PEFT_AVAILABLE` does the same for `peft`, and `hf_datasets` is `None` when the `datasets`
library is missing. Every cell that needs a *pretrained model* is additionally guarded against download
failures: if you are offline and the model is not yet in `DATA_DIR/hf_cache`, the cell prints a
`[skipped]` note and the notebook carries on with the parts that do not need the model (the BPE
tokenizer, word2vec, the LoRA layer, the DPO loss and all of the exercises still run). The helpers at
the end of the setup cell implement that policy once, so that the rest of the notebook stays readable:
`hf_load(loader, name)` wraps `loader.from_pretrained(name, cache_dir=...)` in a `try/except` and
returns `None` on failure, `skip(reason)` prints a uniform note, and `count_params` is used throughout
Sections 4 and 5. Note the `cache_dir` argument: by default Hugging Face stores models in your home
directory; we redirect everything into the course's `data/` folder so that it lives next to the other
datasets and can be deleted in one go.

## 1. Tokenization: from characters to subwords

### 1.1 Characters, words, or something in between?

A neural network consumes numbers, so the first design decision of any language model is how to chop a
string into a sequence of integers from a fixed **vocabulary**. In Notebook 08 · Sequence Models: RNNs,
LSTMs and GRUs and Notebook 09 · Attention and Transformers we used the simplest possible scheme, one
integer per character. Character vocabularies are tiny (a few dozen
to a few thousand symbols), they never encounter an unknown symbol, and they can spell any word. Their
weakness is *length*: a sentence of 20 words becomes a sequence of roughly 100 characters, and since
attention costs grow quadratically in sequence length and the model must learn spelling before it can
learn meaning, character models are slow to train and slow to run.

Word-level vocabularies sit at the opposite extreme. Sequences are short and each token carries meaning,
but the vocabulary explodes: English has hundreds of thousands of word forms, and morphologically rich
languages such as Finnish or Turkish have practically unbounded vocabularies. Any word not seen during
training becomes a single `<unk>` token, which throws away information, and the embedding matrix grows
to $|V| \times d$ parameters, most of which are updated only a handful of times.

**Subword tokenization** is the compromise that every modern language model uses. Frequent words stay
whole (`the`, `model`), rare words are split into frequent pieces (`token` + `ization`), and in the
worst case a word is spelled out from single characters or bytes, so there is never an unknown token.
The vocabulary size becomes a hyperparameter (typically 30k to 250k) that trades off sequence length
against embedding-matrix size. The three families you will meet are **Byte-Pair Encoding** (BPE, used by
GPT-2/3/4, Llama, Qwen), **WordPiece** (BERT, DistilBERT) and the **Unigram** model (T5, many
SentencePiece models). Their vocabularies differ, but all three are learned from data with the same
goal: a set of pieces that makes the training corpus short while keeping every word spellable. BPE and
WordPiece reach it bottom-up, starting from characters and adding pieces; Unigram works top-down,
starting from a large candidate set and pruning it.

| Scheme | Vocabulary | Sequence length | Unknown words | Learns meaning from |
|---|---|---|---|---|
| characters | ~100 | very long | never | characters (slow) |
| words | 100k+ | short | frequent (`<unk>`) | whole words only |
| subwords | 30k–250k (chosen) | medium | never | word pieces |

> **Key idea.** Tokenization is a reversible compression of text into a sequence over a fixed vocabulary;
> subword methods pick the vocabulary that makes the training corpus short while keeping every word
> spellable.

### 1.2 Byte-Pair Encoding from scratch

BPE (Sennrich, Haddow and Birch, 2016, adapting a 1994 compression algorithm) is disarmingly simple:

1. Split the corpus into words and represent each word as a sequence of characters plus an
   end-of-word marker `</w>` (so that `er` at the end of `faster` and `er` inside `ermine` are
   different symbols).
2. Count how often each adjacent pair of symbols occurs across the corpus (weighted by word frequency).
3. Merge the most frequent pair into a single new symbol everywhere, and add the new symbol to the
   vocabulary.
4. Repeat steps 2–3 for a fixed number of merges. The ordered list of merges *is* the tokenizer.

To encode a new word we apply the learned merges in the order they were learned. Because every merge
adds exactly one symbol, the final vocabulary size is `#characters + #merges`, so the number of merges
is the knob that sets the vocabulary size. Let us implement this on Tiny Shakespeare. The download is
guarded with a built-in fallback, exactly as in Notebooks 08 and 09; the same text serves for word2vec
in Section 2.

In [ ]:
TINY_SHAKESPEARE_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

FALLBACK_TEXT = """First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.

Shall I compare thee to a summer's day?
Thou art more lovely and more temperate:
Rough winds do shake the darling buds of May,
And summer's lease hath all too short a date:
Sometime too hot the eye of heaven shines,
And often is his gold complexion dimm'd;
And every fair from fair sometime declines,
By chance or nature's changing course untrimm'd;
But thy eternal summer shall not fade
Nor lose possession of that fair thou owest;
Nor shall Death brag thou wander'st in his shade,
When in eternal lines to time thou growest:
So long as men can breathe or eyes can see,
So long lives this and this gives life to thee.
"""


def load_tiny_shakespeare(path=DATA_DIR / "tinyshakespeare.txt"):
    """Return (text, source) where source is 'cached', 'downloaded' or 'fallback'."""
    if path.exists():
        return path.read_text(encoding="utf-8"), "cached"
    try:
        import urllib.request
        with urllib.request.urlopen(TINY_SHAKESPEARE_URL, timeout=20) as r:
            text = r.read().decode("utf-8")
        path.write_text(text, encoding="utf-8")
        return text, "downloaded"
    except Exception as e:
        print(f"Download failed ({type(e).__name__}): using the built-in fallback text "
              f"(results in Sections 1-2 will be less interesting).")
        return FALLBACK_TEXT * 8, "fallback"


shakespeare, source = load_tiny_shakespeare()
print(f"Tiny Shakespeare ({source}): {len(shakespeare):,} characters, "
      f"{len(shakespeare.split()):,} whitespace-separated words")
print(shakespeare[:180])

The implementation below keeps the corpus as a dictionary that maps each distinct word (as a tuple of
symbols) to its frequency. This is the standard trick that makes BPE tractable: the corpus has hundreds
of thousands of word *tokens* but only a few thousand distinct word *types*, so each merge touches a
few thousand entries rather than the whole text. We record, after every merge, the vocabulary size and
the average number of tokens per word so that we can plot how the segmentation improves as the
vocabulary grows.

In [ ]:
END = "</w>"   # end-of-word marker


def bpe_init(text):
    """Word-frequency dictionary: tuple of symbols (chars + END) -> count."""
    words = re.findall(r"[a-z']+|[^\sa-z']", text.lower())   # words, or single punctuation marks
    freqs = Counter(words)
    return {tuple(w) + (END,): c for w, c in freqs.items()}


def bpe_pair_counts(word_freqs):
    """Count adjacent symbol pairs, weighted by word frequency."""
    pairs = Counter()
    for symbols, freq in word_freqs.items():
        for a, b in zip(symbols, symbols[1:]):
            pairs[(a, b)] += freq
    return pairs


def bpe_merge(pair, word_freqs):
    """Replace every occurrence of `pair` (a, b) by the merged symbol a+b."""
    a, b = pair
    merged = a + b
    out = {}
    for symbols, freq in word_freqs.items():
        new, i = [], 0
        while i < len(symbols):
            if i < len(symbols) - 1 and symbols[i] == a and symbols[i + 1] == b:
                new.append(merged); i += 2
            else:
                new.append(symbols[i]); i += 1
        key = tuple(new)
        out[key] = out.get(key, 0) + freq
    return out


def learn_bpe(text, num_merges, log_every=0):
    """Learn `num_merges` BPE merges. Returns (merges, vocab, history)."""
    word_freqs = bpe_init(text)
    n_words = sum(word_freqs.values())
    vocab = set(s for w in word_freqs for s in w)
    merges, history = [], []
    for step in range(1, num_merges + 1):
        pairs = bpe_pair_counts(word_freqs)
        if not pairs:
            break
        best = max(pairs, key=pairs.get)
        word_freqs = bpe_merge(best, word_freqs)
        merges.append(best)
        vocab.add(best[0] + best[1])
        tokens_per_word = sum(len(w) * f for w, f in word_freqs.items()) / n_words
        history.append((step, len(vocab), tokens_per_word))
        if log_every and step % log_every == 0:
            print(f"merge {step:4d}: {best[0]!r} + {best[1]!r} -> {best[0] + best[1]!r}   "
                  f"(count {pairs[best]:,}, vocab {len(vocab)}, {tokens_per_word:.2f} tokens/word)")
    return merges, vocab, history


BPE_CHARS = 80_000 if FAST_RUN else 300_000
NUM_MERGES = 300 if FAST_RUN else 1000
t0 = time.time()
merges, bpe_vocab, history = learn_bpe(shakespeare[:BPE_CHARS], NUM_MERGES, log_every=NUM_MERGES // 10)
print(f"\nlearned {len(merges)} merges in {time.time() - t0:.1f}s; vocabulary size {len(bpe_vocab)}")
print("first ten merges:", merges[:10])

The first merges are exactly what you would expect from English text: the end-of-word marker attaches
to the most common final letters (`e</w>`, `s</w>`, `d</w>`), then frequent function words such as
`the` and `and` get assembled, and only later do longer content words appear. The log line shows the
raw pair count that motivated each merge; note how quickly it falls, which is why the marginal benefit
of a larger vocabulary shrinks.

Let us plot the two quantities we recorded. The vocabulary size grows by exactly one per merge (a
straight line, by construction), while the average number of tokens per word falls steeply at first and
then flattens. That second curve is the one that matters: it is the compression the tokenizer achieves,
and its knee is a reasonable heuristic for choosing the vocabulary size of a small model.

In [ ]:
steps = [h[0] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
axes[0].plot(steps, [h[1] for h in history])
axes[0].set_xlabel("number of merges"); axes[0].set_ylabel("vocabulary size")
axes[0].set_title("Vocabulary growth")
axes[1].plot(steps, [h[2] for h in history], color="C1")
axes[1].set_xlabel("number of merges"); axes[1].set_ylabel("tokens per word")
axes[1].set_title("Compression achieved by BPE")
plt.tight_layout(); plt.show()
print(f"tokens/word: {history[0][2]:.2f} after 1 merge -> {history[-1][2]:.2f} after {len(history)} merges")

To *use* the tokenizer we need an encoder that applies the learned merges to new text and a decoder that
undoes the segmentation. Encoding a word means repeatedly finding, among the adjacent pairs currently in
the word, the one with the *lowest merge rank* (learned earliest) and merging it, until no pair in the
word has a rank. Because we only ever consult the merge table, words that never appeared in the corpus
are handled naturally: they simply end up in smaller pieces. Decoding concatenates the tokens and turns
the end-of-word marker back into a space.

In [ ]:
merge_ranks = {pair: i for i, pair in enumerate(merges)}


def bpe_encode_word(word, ranks=merge_ranks):
    """Segment one word by applying the learned merges in rank order."""
    symbols = list(word) + [END]
    while len(symbols) > 1:
        candidates = [(ranks.get((a, b), math.inf), i) for i, (a, b) in enumerate(zip(symbols, symbols[1:]))]
        rank, i = min(candidates)
        if rank == math.inf:          # no learnable pair left
            break
        symbols = symbols[:i] + [symbols[i] + symbols[i + 1]] + symbols[i + 2:]
    return symbols


def bpe_encode(text, ranks=merge_ranks):
    """Tokenize a string into BPE tokens (lower-cased, punctuation as separate words)."""
    tokens = []
    for word in re.findall(r"[a-z']+|[^\sa-z']", text.lower()):
        tokens.extend(bpe_encode_word(word, ranks))
    return tokens


def bpe_decode(tokens):
    """Undo the segmentation: concatenate tokens and turn end-of-word markers into spaces."""
    return "".join(tokens).replace(END, " ").strip()


sample = "The tokenization of unfamiliar words, like antidisestablishmentarianism, still works."
tokens = bpe_encode(sample)
print(f"{len(tokens)} tokens:", tokens)
print("decoded:", bpe_decode(tokens))

# how the segmentation of a few words evolves with the number of merges
words_to_watch = ["the", "citizens", "unfamiliar", "shakespeare"]
for k in [0, 20, 100, len(merges)]:
    ranks_k = {m: i for i, m in enumerate(merges[:k])}
    segs = ["|".join(bpe_encode_word(w, ranks_k)) for w in words_to_watch]
    print(f"{k:5d} merges: " + "   ".join(segs))

With no merges every word is spelled out; after a few dozen merges common words such as `the` are
single tokens, and after all merges the corpus-typical word `citizens` is one or two tokens while the
never-seen `unfamiliar` and `shakespeare` are split into meaningful pieces. Exercise 1 asks you to make
the decoder robust to the remaining edge cases (characters outside the training alphabet, and the
whitespace that our lower-casing regular expression discards).

### 1.3 WordPiece, Unigram and SentencePiece

BPE has two well-known relatives. **WordPiece** (Schuster and Nakajima, 2012; used by BERT) follows the
same greedy merge loop but chooses the pair that maximizes the *likelihood* of the corpus under a
unigram language model, i.e. it merges the pair $(a, b)$ that maximizes
$\text{count}(ab) / (\text{count}(a)\,\text{count}(b))$ rather than the raw count. Its vocabulary
marks word-internal pieces with a `##` prefix (`token`, `##ization`) instead of an end-of-word symbol,
and at encoding time it uses a greedy longest-match-first rule rather than replaying merges.
The **Unigram** model (Kudo, 2018) works in the opposite direction: it starts with a very large
candidate vocabulary and repeatedly *removes* the pieces whose removal hurts the corpus likelihood least,
which lets it assign probabilities to alternative segmentations of the same word (useful for
regularization).

**SentencePiece** (Kudo and Richardson, 2018) is not a third algorithm but a *library* that trains BPE
or Unigram tokenizers directly on raw text without a pre-tokenization step. It treats whitespace as an
ordinary character (rendered as `▁`), so that the tokenizer is reversible and language-agnostic, which
matters for Japanese or Chinese where there are no spaces to split on. Modern GPT-style models use
**byte-level BPE** instead: the base alphabet is the 256 byte values, so any Unicode string can be
encoded without an unknown token, at the price of splitting rare characters into several bytes.

### 1.4 Hugging Face tokenizers

Writing tokenizers by hand is instructive, but in practice you must use the *exact* tokenizer a
pretrained model was trained with, including its special tokens and normalization rules: a model's
embedding matrix only knows its own vocabulary. `AutoTokenizer.from_pretrained(name)` downloads the
tokenizer files of a model from the Hugging Face Hub (a few hundred kilobytes) and returns a fast
Rust-backed tokenizer. Let us load the tokenizers of the two models we use throughout: DistilBERT
(WordPiece, uncased) and DistilGPT-2 (byte-level BPE).

In [ ]:
bert_tok = hf_load(AutoTokenizer, "distilbert-base-uncased")
gpt2_tok = hf_load(AutoTokenizer, "distilgpt2")

if bert_tok is not None:
    # DistilBERT has no segment embeddings, so ask for input_ids + attention_mask only (recent
    # transformers versions otherwise also return BERT-style token_type_ids for this tokenizer).
    bert_tok.model_input_names = ["input_ids", "attention_mask"]
    text = "Transformers make tokenization look easy, don't they?"
    enc = bert_tok(text)
    print("keys:", list(enc.keys()))
    print("input_ids     :", enc["input_ids"])
    print("tokens        :", bert_tok.convert_ids_to_tokens(enc["input_ids"]))
    print("attention_mask:", enc["attention_mask"])
    print("decoded       :", bert_tok.decode(enc["input_ids"]))
    print("vocab size    :", len(bert_tok), "| special tokens:", bert_tok.all_special_tokens)
    # a batch: padding to the longest sentence, truncation to max_length, PyTorch tensors
    sentences = ["A short one.",
                 "A somewhat longer sentence that will need to be truncated if the limit is small.",
                 "Medium length, no truncation needed here."]
    batch = bert_tok(sentences, padding=True, truncation=True, max_length=12, return_tensors="pt")
    print("input_ids shape:", tuple(batch["input_ids"].shape))
    for ids, mask in zip(batch["input_ids"], batch["attention_mask"]):
        print(f"{bert_tok.decode(ids):70s} mask={mask.tolist()}")
    # offsets: which characters does each token cover?
    text = "Tokenization isn't trivial: 2026 A.D."
    enc = bert_tok(text, return_offsets_mapping=True)
    print("\ntoken -> character span")
    for tok, (s, e) in zip(bert_tok.convert_ids_to_tokens(enc["input_ids"]), enc["offset_mapping"]):
        print(f"  {tok:>8s}  [{s:2d},{e:2d})  {text[s:e]!r}")
else:
    skip("tokenizer demos need the DistilBERT tokenizer")

Three things to notice in the first block of output. First, the tokenizer *added* tokens: `[CLS]` at
the start and `[SEP]` at the end. BERT-style encoders were pretrained with these **special tokens**
and expect them at inference time; the classification head we fine-tune later reads the hidden state
at the `[CLS]` position. Second, the word `tokenization` was split into `token` + `##ization` with
WordPiece's `##` continuation marker, and the contraction `don't` became `don`, `'`, `t`. Third, the
output contains an **attention mask** of ones (recent library versions would also return
`token_type_ids`, the segment ids BERT uses for sentence-pair inputs; DistilBERT has no segment
embeddings, which is why we set `model_input_names` above, and the collate function of Section 4 drops
the key again should it ever appear). The mask's purpose is visible in the second block, where we
tokenize a *batch* of sentences of different lengths: the tokenizer pads the short ones with `[PAD]`
tokens so that the batch is a rectangular tensor, and the mask records which positions are real (`1`)
and which are padding (`0`), so that attention can ignore the padding.

`padding=True` pads to the longest sequence *in the batch* (called dynamic padding, which we exploit in
Section 4), `truncation=True` cuts sequences that exceed `max_length` (keeping the `[SEP]` token), and
`return_tensors="pt"` gives PyTorch tensors instead of lists. The **offset mapping** tells us which
characters of the original string each token covers; it is what you need for token-classification
tasks (named-entity recognition, question answering) where predictions have to be mapped back to
spans of the text. Special tokens have the empty span `[0, 0)`.

The GPT-2 tokenizer looks different because it is byte-level BPE with no pre-tokenization on
whitespace: the space *belongs to the following word* and is rendered as `Ġ`. The consequence, which
surprises everyone once, is that `"hello"` and `" hello"` are different tokens with different
embeddings, and that prompts should therefore not end with a trailing space. GPT-2 also has no `[CLS]`
or padding token; it only knows `<|endoftext|>`, so for batched generation we must designate a pad
token ourselves (the usual choice is to reuse the end-of-text token).

In [ ]:
if gpt2_tok is not None:
    print("tokens :", gpt2_tok.tokenize("Hello world! Tokenization isn't trivial."))
    print('"hello" ->', gpt2_tok.encode("hello"), '  " hello" ->', gpt2_tok.encode(" hello"))
    print("vocab size:", len(gpt2_tok), "| special tokens:", gpt2_tok.all_special_tokens,
          "| pad token:", gpt2_tok.pad_token)
    gpt2_tok.pad_token = gpt2_tok.eos_token       # needed for batched generation later
else:
    skip("GPT-2 tokenizer demo needs the DistilGPT-2 tokenizer")

### 1.5 Tokenization pitfalls

Because the tokenizer is fixed before training, its quirks become the model's quirks. The following
are worth knowing.

- **Numbers.** Vocabularies contain frequent numbers as single tokens (`2024`, `100`) but split rare
  ones into arbitrary chunks (`12345678` becomes `123` + `456` + `78`), which is one reason language
  models are bad at arithmetic: the digits are not aligned. Newer tokenizers deliberately split numbers
  into single digits or groups of three.
- **Whitespace and case.** An uncased tokenizer folds `HELLO`, `Hello` and `hello` into one token,
  so a model built on it cannot detect shouting. A cased byte-level tokenizer keeps them apart, at the
  cost of three separate embeddings that must each learn what "hello" means. Leading spaces, tabs and
  newlines are all distinct tokens for GPT-style models, which is why code with mixed indentation
  tokenizes wastefully.
- **Multilingual text.** A vocabulary learned mostly on English spends few pieces on other scripts.
  Japanese or Hindi text may cost three to five times as many tokens per character as English, which
  means shorter effective context windows and higher API prices for the same content. BERT's uncased
  vocabulary even maps many characters to `[UNK]`.
- **Tokenizer/model mismatch.** Loading a model with a different tokenizer (even a similar one) silently
  produces garbage because the token ids index a different embedding table. Always load both from the
  same checkpoint name.

The cell below makes the first three points concrete by counting tokens for a handful of strings under
both tokenizers.

In [ ]:
probe_strings = ["2024", "12345678", "The price is $1,234.56.", "HELLO hello Hello",
                 "naïve café résumé", "こんにちは世界", "def f(x):\n    return x ** 2"]
if bert_tok is not None and gpt2_tok is not None:
    print(f"{'string':32s} {'chars':>5s} {'BERT':>5s} {'GPT2':>5s}   BERT tokens")
    for s in probe_strings:
        b = bert_tok.tokenize(s); g = gpt2_tok.tokenize(s)
        print(f"{s!r:32s} {len(s):5d} {len(b):5d} {len(g):5d}   {b[:8]}")
else:
    skip("tokenizer comparison needs both Hugging Face tokenizers")

Read the table with the bullet points above in mind: the eight-digit number is chopped into pieces, the
three spellings of `hello` collapse to one WordPiece token but three GPT-2 tokens, the accented words
cost several tokens each, and the Japanese greeting is either a long run of single characters or, for
BERT, partly `[UNK]`. Exercise 2 extends this comparison to more languages.

## 2. Word embeddings: word2vec from scratch

### 2.1 From one-hot vectors to dense embeddings

Once text is a sequence of integer ids, we need vectors. The naive choice is the **one-hot** encoding,
$e_i \in \{0,1\}^{|V|}$ with a single one at position $i$. It has two problems. It is enormous
(30k dimensions per token) and, worse, it is *uninformative*: every pair of distinct one-hot vectors is
orthogonal, so `cat` is exactly as far from `dog` as it is from `carburettor`. A **dense embedding**
instead maps each id to a learned vector $v_i \in \mathbb{R}^d$ with $d \ll |V|$ (typically 64–1024),
stored as the rows of a matrix $E \in \mathbb{R}^{|V| \times d}$. Looking up row $i$ is mathematically
the product $e_i^\top E$, but `nn.Embedding` implements it as an index operation, which is why it is
cheap. The point of an embedding is that *similar tokens should have similar vectors*, and the
question is where that similarity should come from.

In [ ]:
V, d = 8, 4
emb = nn.Embedding(V, d)
ids = torch.tensor([2, 5])
one_hot = F.one_hot(ids, num_classes=V).float()                 # (2, V)
lookup = emb(ids)                                               # (2, d), index lookup
matmul = one_hot @ emb.weight                                   # (2, d), the equivalent product
print("lookup == one_hot @ E :", torch.allclose(lookup, matmul))
print("cosine similarity between distinct one-hot vectors:",
      F.cosine_similarity(F.one_hot(torch.tensor(2), V).float(), F.one_hot(torch.tensor(5), V).float(), dim=0).item())

### 2.2 The skip-gram objective with negative sampling

The **distributional hypothesis** (Harris, 1954; "you shall know a word by the company it keeps",
Firth, 1957) supplies the training signal: words that occur in similar contexts should get similar
vectors. **word2vec** (Mikolov et al., 2013) turns this into a prediction task. In the **skip-gram**
variant, each word $w_c$ at position $t$ is the *center* and the words in a window of $\pm m$ positions
around it are the *contexts* $w_o$. The model has two embedding tables, $v_w$ for center words and
$u_w$ for context words, and scores a (center, context) pair by the dot product $u_o^\top v_c$. With a
softmax over the vocabulary,

$$
p(w_o \mid w_c) = \frac{\exp(u_o^\top v_c)}{\sum_{w \in V} \exp(u_w^\top v_c)},
$$

and the training objective is the log-likelihood of all observed pairs. The denominator is the
problem: every training step would need $|V|$ dot products. **Negative sampling** replaces the
$|V|$-way classification by a binary one. For each observed pair $(c, o)$ we draw $K$ "negative"
words $w_1, \dots, w_K$ from a noise distribution $P_n$ and ask the model to say *yes* to the true
context and *no* to the negatives:

$$
\mathcal{L}(c, o) = -\log \sigma(u_o^\top v_c) - \sum_{k=1}^{K} \log \sigma(-u_{w_k}^\top v_c),
\qquad w_k \sim P_n(w) \propto \text{count}(w)^{3/4}.
$$

Here $\sigma$ is the logistic sigmoid, and the exponent $3/4$ flattens the unigram distribution so that
rare words are sampled a little more often than their frequency suggests (an empirical choice from the
paper). Minimizing $\mathcal{L}$ pushes $v_c$ towards $u_o$ and away from the $u_{w_k}$; because
frequent co-occurrences are visited many times, words that share contexts converge to similar
$v$-vectors. Two more practical details from the original implementation matter: **subsampling** of
very frequent words (each occurrence of word $w$ is discarded with probability
$1 - \sqrt{t / f(w)}$, with $f(w)$ the relative frequency and $t$ between $10^{-5}$ and $10^{-3}$,
larger for small corpora), which removes uninformative `the`/`and` pairs and effectively widens the
window, and the use of the *center* table $v$ as the final embedding.

> **Going deeper.** Levy and Goldberg (2014) showed that skip-gram with negative sampling implicitly
> factorizes a shifted pointwise-mutual-information matrix, $u_o^\top v_c \approx \text{PMI}(c, o) -
> \log K$. This explains why word2vec and older count-based methods such as GloVe behave so similarly.

### 2.3 Preparing the corpus

We tokenize Tiny Shakespeare into lower-case words, keep words that occur at least `MIN_COUNT` times,
apply subsampling, and generate (center, context) pairs for a window of $\pm$`WINDOW` positions.
Under `FAST_RUN` we use the first 60k words with a smaller batch size (so that five epochs still give
a few thousand optimizer steps); the full run uses the whole corpus.

In [ ]:
def tokenize_words(text):
    return re.findall(r"[a-z']+", text.lower())


all_words = tokenize_words(shakespeare)
N_WORDS = 60_000 if FAST_RUN else len(all_words)
MIN_COUNT = 3 if FAST_RUN else 5
WINDOW = 2 if FAST_RUN else 3
words = all_words[:N_WORDS]

word_counts = Counter(words)
w2v_vocab = [w for w, c in word_counts.most_common() if c >= MIN_COUNT]
w2i = {w: i for i, w in enumerate(w2v_vocab)}
ids = np.array([w2i[w] for w in words if w in w2i])           # drop rare words
freq = np.bincount(ids, minlength=len(w2v_vocab)) / len(ids)

# subsampling of frequent words (Mikolov et al., 2013); t = 1e-3 suits a corpus this small
t_sub = 1e-3
keep_prob = np.minimum(1.0, np.sqrt(t_sub / freq[ids]))
rng = np.random.default_rng(SEED)
ids_sub = ids[rng.random(len(ids)) < keep_prob]

# (center, context) pairs
centers, contexts = [], []
for offset in range(1, WINDOW + 1):
    centers.append(ids_sub[offset:]);  contexts.append(ids_sub[:-offset])   # context before center
    centers.append(ids_sub[:-offset]); contexts.append(ids_sub[offset:])    # context after center
pairs = torch.tensor(np.stack([np.concatenate(centers), np.concatenate(contexts)], axis=1))

# noise distribution for negative sampling: unigram^0.75
noise_dist = torch.tensor(np.bincount(ids, minlength=len(w2v_vocab)) ** 0.75, dtype=torch.float)
noise_dist = (noise_dist / noise_dist.sum()).to(device)

print(f"{len(words):,} words -> vocab {len(w2v_vocab):,} (min count {MIN_COUNT}) -> "
      f"{len(ids):,} in-vocabulary tokens -> {len(ids_sub):,} after subsampling -> {len(pairs):,} pairs")
print("most frequent:", w2v_vocab[:12])

### 2.4 The model and the training loop

The model is two embedding tables and the loss above, written with batched operations. For a batch of
$B$ pairs we look up $v_c$ (shape `(B, d)`), $u_o$ (`(B, d)`) and the $K$ negatives $u_{w_k}$
(`(B, K, d)`), compute the positive scores with an element-wise product and sum, and the negative
scores with a batched matrix–vector product. `F.logsigmoid` is the numerically stable way to compute
$\log\sigma(x)$ (it never evaluates $\log$ of a tiny number). We initialize the context table to zero
and the center table to small uniform values, as the reference implementation does; with a zero
context table the first gradient steps are well-behaved. Negatives are drawn with
`torch.multinomial` from the $3/4$-power unigram distribution, on the device, once per batch.

In [ ]:
class SkipGramNS(nn.Module):
    """Skip-gram word2vec with negative sampling."""

    def __init__(self, vocab_size, dim):
        super().__init__()
        self.in_emb = nn.Embedding(vocab_size, dim)     # v: center-word vectors (the final embedding)
        self.out_emb = nn.Embedding(vocab_size, dim)    # u: context-word vectors
        nn.init.uniform_(self.in_emb.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.out_emb.weight)

    def forward(self, center, context, negatives):
        v = self.in_emb(center)                                     # (B, d)
        u_pos = self.out_emb(context)                               # (B, d)
        u_neg = self.out_emb(negatives)                             # (B, K, d)
        pos = F.logsigmoid((u_pos * v).sum(-1))                     # (B,)
        neg = F.logsigmoid(-torch.bmm(u_neg, v.unsqueeze(2)).squeeze(2)).sum(1)   # (B,)
        return -(pos + neg).mean()


def train_word2vec(pairs, vocab_size, dim=64, epochs=1, batch_size=1024, K=5, lr=3e-3):
    torch.manual_seed(SEED)
    model = SkipGramNS(vocab_size, dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    pairs = pairs.to(device)
    losses = []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        perm = torch.randperm(len(pairs), device=device)
        running, n_batches = 0.0, 0
        model.train()
        for start in range(0, len(pairs), batch_size):
            batch = pairs[perm[start:start + batch_size]]
            negatives = torch.multinomial(noise_dist, batch.size(0) * K, replacement=True).view(-1, K)
            loss = model(batch[:, 0], batch[:, 1], negatives)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            running += loss.item(); n_batches += 1
        losses.append(running / n_batches)
        print(f"epoch {epoch:2d} | loss {losses[-1]:.4f} | {time.time() - t0:.1f}s")
    return model, losses


W2V_EPOCHS = 5
W2V_BATCH = 256 if FAST_RUN else 1024
w2v, w2v_losses = train_word2vec(pairs, len(w2v_vocab), dim=64, epochs=W2V_EPOCHS, batch_size=W2V_BATCH)

The loss starts near $(K+1)\log 2 \approx 4.16$ (a model that is indifferent to every pair pays
$\log 2$ per binary decision) and falls as the tables organize themselves. To see what was learned we
look up nearest neighbours by **cosine similarity**, $\cos(a, b) = a^\top b / (\|a\|\,\|b\|)$, which
ignores the vector norm (norms mostly track word frequency) and keeps only the direction. One
preprocessing step is essential on a small corpus: we **subtract the mean vector** before
normalizing. Negative sampling pushes every center vector away from the context vectors of the
frequent words that dominate the noise distribution, so all embeddings acquire a large *shared*
component; without centering, every pair of words has a cosine close to 1 and the neighbours are
meaningless. Removing the common mean direction (the "all-but-the-top" post-processing of Mu and
Viswanath, 2018) exposes the differences between words, which is what we care about.

The same cell also tests the famous **analogy** property, $v_{\text{king}} - v_{\text{man}} +
v_{\text{woman}} \approx v_{\text{queen}}$, which emerges because directions in the embedding space
encode relations that are shared across many word pairs. It is a consequence of the linear structure
of the objective, not something the model was told to do. On a corpus as small as Shakespeare the
effect is weak, so treat the analogy results as an illustration of the *method* (we search for the
nearest neighbour of the offset vector, excluding the three query words) rather than as a guaranteed
result.

In [ ]:
@torch.no_grad()
def embedding_matrix(model):
    E = model.in_emb.weight.detach()
    E = E - E.mean(0, keepdim=True)       # remove the common direction shared by all words
    return F.normalize(E, dim=1)          # unit-length rows -> dot product == cosine similarity


def nearest(word, k=6, E=None):
    E = embedding_matrix(w2v) if E is None else E
    if word not in w2i:
        return f"'{word}' not in vocabulary"
    sims = E @ E[w2i[word]]
    sims[w2i[word]] = -1.0                # exclude the query itself
    top = sims.topk(k)
    return [(w2v_vocab[i], round(s, 3)) for s, i in zip(top.values.tolist(), top.indices.tolist())]


for w in ["king", "love", "death", "night", "sword", "lady"]:
    print(f"{w:>6s} -> {nearest(w)}")


# analogies:  a : b :: c : ?   ->   nearest neighbours of  v_b - v_a + v_c  (query words excluded)
@torch.no_grad()
def analogy(a, b, c, k=5):
    """a : b :: c : ?   ->   nearest neighbours of  v_b - v_a + v_c."""
    if any(w not in w2i for w in (a, b, c)):
        return "some query word is not in the vocabulary"
    E = embedding_matrix(w2v)
    target = F.normalize(E[w2i[b]] - E[w2i[a]] + E[w2i[c]], dim=0)
    sims = E @ target
    for w in (a, b, c):
        sims[w2i[w]] = -1.0
    top = sims.topk(k)
    return [(w2v_vocab[i], round(s, 3)) for s, i in zip(top.values.tolist(), top.indices.tolist())]


print("man : king :: woman : ?   ->", analogy("man", "king", "woman"))
print("he : his :: she : ?       ->", analogy("he", "his", "she"))
print("king : kings :: lord : ?  ->", analogy("king", "kings", "lord"))

On the full corpus the neighbours are sensible: royal titles cluster together, `love` sits near other
abstract nouns of affection, and `sword` near weapons and violence. Under `FAST_RUN` (60k words, a
vocabulary of ~2k types) they are noisier, and on the fallback text they are meaningless, since a few hundred words
cannot pin down a 64-dimensional geometry; this is your first encounter with the fact that embeddings
need *data* more than they need clever architectures. The analogy queries behave the same way: on the
full corpus the top candidates are usually plausible (a royal or feminine noun for the first query, a
possessive for the second), on tiny corpora they are noise.

A two-dimensional projection gives a qualitative picture of the geometry. We project the embeddings of
a hand-picked set of words onto their first two principal components (via `torch.pca_lowrank`, which
computes a truncated SVD). Two dimensions cannot do justice to 64, but related words should at least
land near each other.

In [ ]:
probe_words = ["king", "queen", "prince", "duke", "lord", "lady", "father", "mother", "son", "daughter",
               "love", "hate", "death", "life", "blood", "sword", "war", "peace", "heaven", "hell",
               "day", "night", "sun", "moon", "good", "sweet", "fair", "true", "come", "go", "speak", "hear"]
probe_words = [w for w in probe_words if w in w2i]
E_probe = embedding_matrix(w2v)[[w2i[w] for w in probe_words]].cpu()
E_centered = E_probe - E_probe.mean(0, keepdim=True)
_, _, V2 = torch.pca_lowrank(E_centered, q=2, center=False)   # V2: (64, 2) principal directions
proj = (E_centered @ V2).cpu().numpy()              # E_probe was moved to the CPU above

plt.figure(figsize=(7, 5))
plt.scatter(proj[:, 0], proj[:, 1], s=18)
for (x, y), w in zip(proj, probe_words):
    plt.annotate(w, (x, y), fontsize=8, xytext=(3, 3), textcoords="offset points")
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.title("word2vec embeddings (PCA projection)")
plt.tight_layout(); plt.show()

### 2.5 From static to contextual embeddings

word2vec assigns *one* vector per word type. That is a real limitation: `bank` gets a single vector
that must somehow serve both the river bank and the savings bank, and `not good` has no way to modify
`good`. The next step in the history of NLP (ELMo in 2018, then BERT and GPT) was the **contextual
embedding**: instead of a table lookup, the vector for a token is the output of a deep network that has
read the whole sentence, so the same word gets different vectors in different contexts. Pretrained
Transformers are, from this point of view, giant contextual embedding functions, and everything in the
rest of this notebook is about reusing them. We verify the claim numerically in Section 3.1 by comparing
DistilBERT's vectors for `bank` in two sentences.

> **Key idea.** Static embeddings learn one vector per word from co-occurrence statistics; contextual
> embeddings compute a vector per *occurrence* by running the sentence through a pretrained network.

Notebook 10 - part 2 of 4 (using pretrained models, fine-tuning by hand and with Trainer)

## 3. Using pretrained models

### 3.1 `AutoModel`: what a pretrained encoder returns

The Hugging Face `transformers` library exposes thousands of checkpoints through a handful of
**Auto classes**. `AutoModel.from_pretrained(name)` reads the model's configuration file, builds the
matching architecture (a `DistilBertModel`, a `GPT2Model`, ...) and loads the weights; the
task-specific variants (`AutoModelForMaskedLM`, `AutoModelForSequenceClassification`,
`AutoModelForCausalLM`) add the corresponding head on top. All of them are ordinary `nn.Module`s, so
everything you know from the course applies: `.to(device)`, `.eval()`, `torch.no_grad()`,
`state_dict()`.

Our workhorse encoder is **DistilBERT** (Sanh et al., 2019): a 6-layer, 768-dimensional, 12-head
Transformer encoder with 66M parameters, distilled from BERT-base (12 layers, 110M) to keep about 97%
of its accuracy at 60% of the size. It was pretrained with the masked-language-model objective from
Notebook 09 · Attention and Transformers on English Wikipedia and BookCorpus. The plain `AutoModel`
returns the final hidden states,
one 768-dimensional vector per token, which is precisely the "contextual embedding" of Section 2.5.

In [ ]:
bert = hf_load(AutoModel, "distilbert-base-uncased")
if bert is not None and bert_tok is not None:
    bert = bert.to(device).eval()
    total, _ = count_params(bert)
    print(f"DistilBERT: {total / 1e6:.1f}M parameters, {bert.config.n_layers} layers, "
          f"hidden size {bert.config.dim}, {bert.config.n_heads} heads")
    enc = bert_tok("The quick brown fox jumps over the lazy dog.", return_tensors="pt").to(device)
    with torch.no_grad():
        out = bert(**enc, output_hidden_states=True)
    print("last_hidden_state:", tuple(out.last_hidden_state.shape), "  (batch, tokens, hidden)")
    print("hidden_states    :", len(out.hidden_states), "tensors (embeddings + one per layer)")
    cls_vec = out.last_hidden_state[:, 0]                              # the [CLS] position
    mask = enc["attention_mask"].unsqueeze(-1).float()
    mean_vec = (out.last_hidden_state * mask).sum(1) / mask.sum(1)     # mean over real tokens
    print("CLS pooling:", tuple(cls_vec.shape), " mean pooling:", tuple(mean_vec.shape))

    # the promised check that embeddings are contextual: "bank" in money vs river contexts
    def word_vector(sentence, word):
        enc = bert_tok(sentence, return_tensors="pt").to(device)
        toks = bert_tok.convert_ids_to_tokens(enc["input_ids"][0])
        with torch.no_grad():
            h = bert(**enc).last_hidden_state[0]
        return h[toks.index(word)]

    v_money1 = word_vector("She deposited the cheque at the bank this morning.", "bank")
    v_money2 = word_vector("The bank approved the loan and opened a new account.", "bank")
    v_river = word_vector("They sat on the bank of the river and watched the water.", "bank")
    cos = lambda a, b: F.cosine_similarity(a, b, dim=0).item()
    print(f"cos(bank_money1, bank_money2) = {cos(v_money1, v_money2):.3f}")
    print(f"cos(bank_money1, bank_river)  = {cos(v_money1, v_river):.3f}")
else:
    skip("AutoModel demo needs the DistilBERT weights")

Two **pooling** strategies turn the per-token outputs into one vector per sentence: take the hidden
state at the `[CLS]` position (what BERT's authors intended, and what classification heads use) or
average the hidden states over the real tokens (masking out padding). Mean pooling is usually the
better sentence representation for an encoder that has *not* been fine-tuned, and we use it for
semantic search in Section 3.4.

The second half of the cell checks that the embeddings are contextual: the same word `bank` in a
financial and in a river sentence should get less similar vectors than in two financial sentences.
The two financial `bank`s are indeed closer to each other than either is to the river `bank`; a static
embedding would give a cosine of exactly 1 in both cases. The difference is modest (DistilBERT's
hidden states all share a large common component), but it is systematic.

### 3.2 Fill-mask with the pretrained head

`AutoModelForMaskedLM` adds the pretraining head, a linear layer from the hidden size to the
vocabulary, so we can ask the model the question it was trained on: "which token belongs in the
`[MASK]` position?" The head produces logits of shape `(batch, tokens, vocab)`; we take the row at the
mask position, apply softmax, and read off the top candidates. This is a cheap way to probe what the
model knows about syntax and world knowledge, and it is also the basis of *prompt-based* zero-shot
classification for encoders.

In [ ]:
mlm = hf_load(AutoModelForMaskedLM, "distilbert-base-uncased")
if mlm is not None and bert_tok is not None:
    mlm = mlm.to(device).eval()

    @torch.no_grad()
    def fill_mask(text, k=5):
        enc = bert_tok(text, return_tensors="pt").to(device)
        logits = mlm(**enc).logits                                              # (1, T, V)
        pos = (enc["input_ids"][0] == bert_tok.mask_token_id).nonzero()[0, 0]
        probs = logits[0, pos].softmax(-1)
        top = probs.topk(k)
        return [(bert_tok.decode([i]), round(p, 3)) for p, i in zip(top.values.tolist(), top.indices.tolist())]

    for text in ["The capital of France is [MASK].",
                 "She poured the coffee into a [MASK].",
                 "Deep learning models are trained with gradient [MASK]."]:
        print(f"{text:58s} -> {fill_mask(text)}")
else:
    skip("fill-mask demo needs the DistilBERT masked-LM weights")

### 3.3 Text generation with DistilGPT-2

Decoder-only models are used through `generate`, which implements the decoding strategies from
Notebook 09 (greedy, sampling with temperature, top-k, top-p, beam search, repetition penalties) and
a KV-cache, so that you do not have to write the loop yourself. `distilgpt2` is a 6-layer distillation
of GPT-2 small (82M parameters); its text is fluent at the phrase level and incoherent at the paragraph
level, which is exactly the behaviour of a small language model trained only with next-token
prediction. The generation parameters are worth memorizing:

- `max_new_tokens`: how many tokens to append (prefer it to `max_length`, which counts the prompt).
- `do_sample=False`: greedy decoding (deterministic, repetitive); `do_sample=True` samples.
- `temperature`: divides the logits before softmax; below 1 sharpens, above 1 flattens.
- `top_k` / `top_p`: restrict sampling to the $k$ most likely tokens / the smallest set with
  cumulative probability $p$ (nucleus sampling).
- `repetition_penalty`: down-weights tokens that already appeared; `num_beams` enables beam search.

In [ ]:
gpt2 = hf_load(AutoModelForCausalLM, "distilgpt2")
if gpt2 is not None and gpt2_tok is not None:
    gpt2 = gpt2.to(device).eval()
    prompt = "In the future, machine learning will"
    enc = gpt2_tok(prompt, return_tensors="pt").to(device)
    settings = {"greedy": dict(do_sample=False),
                "sampling T=0.7, top-p 0.9": dict(do_sample=True, temperature=0.7, top_p=0.9),
                "sampling T=1.3, top-k 50": dict(do_sample=True, temperature=1.3, top_k=50)}
    for name, kw in settings.items():
        torch.manual_seed(SEED)
        with torch.no_grad():
            out = gpt2.generate(**enc, max_new_tokens=30, pad_token_id=gpt2_tok.eos_token_id,
                                repetition_penalty=1.2, **kw)
        print(f"[{name}]\n  {gpt2_tok.decode(out[0], skip_special_tokens=True)!r}\n")
else:
    skip("generation demo needs the DistilGPT-2 weights")

Compare the three outputs: greedy decoding is coherent for a few words and then tends to loop (the
`repetition_penalty` softens this), low-temperature nucleus sampling is the usual sweet spot, and a
high temperature produces surprising but increasingly ungrammatical continuations. Every one of these
knobs shapes the *same* probability distribution; none of them adds knowledge that the model lacks.

### 3.4 Sentence embeddings and a tiny semantic search engine

Mean-pooled hidden states give us a vector per sentence, and cosine similarity between such vectors is
a workable notion of *semantic* similarity: two sentences about the same topic land near each other
even if they share no words. That is the core of dense retrieval and of the retrieval-augmented
generation (RAG) systems that ground LLM answers in documents. The cell below embeds a small corpus
and ranks it against a query. Plain DistilBERT was never trained to make sentence vectors comparable,
so this is a weak baseline; dedicated sentence-embedding models (the `sentence-transformers` family,
trained contrastively on sentence pairs) are much better and use exactly the same pooling code.

In [ ]:
corpus = ["The stock market fell sharply after the central bank raised interest rates.",
          "A new vaccine shows strong protection against the seasonal flu.",
          "The football team won the championship after a dramatic penalty shootout.",
          "Researchers trained a neural network to translate between forty languages.",
          "Heavy rain caused flooding in several coastal towns over the weekend.",
          "The company reported record quarterly profits and raised its dividend.",
          "Doctors recommend regular exercise to reduce the risk of heart disease.",
          "The chess grandmaster defended her title in a tense final game."]


@torch.no_grad()
def embed_sentences(texts, batch_size=16):
    """Mean-pooled DistilBERT embeddings, L2-normalized, as a (n, 768) tensor."""
    vecs = []
    for i in range(0, len(texts), batch_size):
        enc = bert_tok(texts[i:i + batch_size], padding=True, truncation=True, max_length=64,
                       return_tensors="pt").to(device)
        h = bert(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        vecs.append(F.normalize((h * m).sum(1) / m.sum(1), dim=1))
    return torch.cat(vecs)


if bert is not None and bert_tok is not None:
    corpus_emb = embed_sentences(corpus)
    for query in ["Which news is about finance?", "A story about sports"]:
        sims = corpus_emb @ embed_sentences([query])[0]
        print(f"query: {query}")
        for s, i in zip(*[t.tolist() for t in sims.topk(3)]):
            print(f"   {s:.3f}  {corpus[i]}")
else:
    skip("semantic search demo needs the DistilBERT weights")

### 3.5 The `pipeline` API

For quick experiments the `pipeline` function bundles tokenizer, model, pre- and post-processing into
one callable: `pipeline("fill-mask")`, `pipeline("text-generation")`, `pipeline("sentiment-analysis")`
and dozens more. Called with only a task name it downloads a default checkpoint, which is convenient
but not reproducible; passing explicit `model` and `tokenizer` objects, as below, is the disciplined
way. Pipelines are ideal for demos and for inference on a handful of inputs; for training, evaluation
on thousands of examples, or custom preprocessing you want the explicit `tokenizer` + `model` code we
have been writing.

In [ ]:
if mlm is not None and pipeline is not None:
    try:
        unmasker = pipeline("fill-mask", model=mlm, tokenizer=bert_tok, device=device)
        for r in unmasker("Paris is the [MASK] of France.", top_k=3):
            print(f"{r['token_str']:>10s}  {r['score']:.3f}   {r['sequence']}")
    except Exception as e:
        skip(f"pipeline failed: {type(e).__name__}: {str(e)[:100]}")
else:
    skip("pipeline demo needs the masked-LM model")

# free the models we no longer need (keep `bert` for later sections)
mlm = gpt2 = None
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

## 4. Fine-tuning for classification, by hand and with `Trainer`

### 4.1 The task and the data

Everything so far used the pretrained models as they are. **Fine-tuning** changes their weights on a
labelled dataset for a downstream task, and it is the single most effective technique in applied NLP:
a pretrained encoder plus a few thousand labelled examples routinely beats a model trained from scratch
on a hundred times more data, because the encoder already knows the language and only has to learn the
task. Our task is binary sentiment classification on **SST-2** (the Stanford Sentiment Treebank, a
GLUE benchmark task): short movie-review sentences labelled positive (1) or negative (0), 67k training
sentences and 872 validation sentences. We load it with the `datasets` library, which caches
downloads in `DATA_DIR/hf_cache`, and take a subset sized by `FAST_RUN`. If the download fails or the
library is missing, we fall back to a small labelled dataset generated inside the notebook (about two
hundred sentences about films built from templates and hand-written examples), so that the data
pipeline and the exercises below run offline.

In [ ]:
def make_fallback_dataset(seed=SEED):
    """216 labelled movie-review sentences generated in the notebook (label 1 = positive)."""
    subjects = ["the film", "this movie", "the story", "the acting", "the script", "the soundtrack",
                "the direction", "the ending", "the cast", "the cinematography"]
    positive = ["is a genuine delight from start to finish", "kept me smiling the whole way through",
                "is smart, warm and beautifully crafted", "deserves every bit of praise it has received",
                "surprised me with its depth and humour", "is easily the best thing I have seen this year",
                "left the audience cheering", "is moving without ever being sentimental",
                "shows real craft and a big heart", "is a small miracle of storytelling"]
    negative = ["is a tedious mess from start to finish", "bored me within the first ten minutes",
                "is clumsy, loud and painfully predictable", "wastes a talented cast on a hollow script",
                "collapses under its own pretensions", "is easily the worst thing I have seen this year",
                "left the audience checking their phones", "is manipulative and never earns its emotions",
                "feels cheap and thrown together", "is a chore to sit through"]
    extra_pos = ["A tender, funny and quietly devastating portrait of a family.",
                 "Gorgeous to look at and even better to listen to.",
                 "The lead performance alone is worth the price of admission.",
                 "An inventive thriller that never once loses its nerve.",
                 "Charming, witty and full of life.", "I laughed, I cried, I want to see it again.",
                 "A rare sequel that improves on the original.", "Thrilling from the first frame to the last."]
    extra_neg = ["A dull, lifeless remake nobody asked for.", "Ninety minutes of my life I will not get back.",
                 "The jokes fall flat and the plot makes no sense.", "Loud, ugly and utterly forgettable.",
                 "Even the outtakes would have been more entertaining.", "A tired formula executed without conviction.",
                 "The dialogue is wooden and the pacing is glacial.", "Painfully long and hopelessly confused."]
    texts, labels = [], []
    for s in subjects:
        for p in positive:
            texts.append(f"{s} {p}."); labels.append(1)
        for n in negative:
            texts.append(f"{s} {n}."); labels.append(0)
    texts += extra_pos + extra_neg
    labels += [1] * len(extra_pos) + [0] * len(extra_neg)
    rng = random.Random(seed)
    order = list(range(len(texts)))
    rng.shuffle(order)
    return [texts[i] for i in order], [labels[i] for i in order]


def load_sentiment_data(n_train, n_val):
    """SST-2 subsets via `datasets` if possible, else the in-notebook fallback. Returns plain lists."""
    if hf_datasets is not None:
        for args in [("stanfordnlp/sst2",), ("glue", "sst2")]:      # same data; the first name works with datasets >= 3
            try:
                ds = hf_datasets.load_dataset(*args, cache_dir=str(HF_CACHE))
                tr = ds["train"].shuffle(seed=SEED).select(range(n_train))
                va = ds["validation"].select(range(min(n_val, len(ds["validation"]))))
                print(f"SST-2 loaded via datasets.load_dataset{args}: {len(tr)} train / {len(va)} val")
                return list(tr["sentence"]), list(tr["label"]), list(va["sentence"]), list(va["label"]), "sst2"
            except Exception as e:
                print(f"could not load {args}: {type(e).__name__}: {str(e)[:80]}")
    else:
        print("`datasets` is not installed.")
    texts, labels = make_fallback_dataset()
    n_tr = int(0.8 * len(texts))
    print(f"-> using the in-notebook fallback dataset: {n_tr} train / {len(texts) - n_tr} val sentences")
    return texts[:n_tr], labels[:n_tr], texts[n_tr:], labels[n_tr:], "fallback"


N_TRAIN = 512 if FAST_RUN else 4000
N_VAL = 256 if FAST_RUN else 1000
train_texts, train_labels, val_texts, val_labels, data_source = load_sentiment_data(N_TRAIN, N_VAL)
print(f"positive share: train {np.mean(train_labels):.2f}, val {np.mean(val_labels):.2f}")
for t, l in list(zip(train_texts, train_labels))[:4]:
    print(f"  [{l}] {t}")

### 4.2 Datasets, dynamic padding and the collate function

We wrap the lists in a minimal `Dataset` that returns raw strings and labels, and do the tokenization
in the DataLoader's **collate function**. Tokenizing per batch rather than up front has one big
advantage: the tokenizer pads each batch only to the length of its *longest* member (**dynamic
padding**), instead of to a global maximum. Since attention cost is quadratic in length and most
sentences are short, this saves a large fraction of compute for free. If Hugging Face tokenizers are
unavailable we substitute a small word-level tokenizer with the same call signature, so that the
pipeline (and the exercises that reuse it) still runs.

In [ ]:
class SimpleWordTokenizer:
    """Word-level stand-in for AutoTokenizer (same call signature) so the pipeline runs offline."""

    def __init__(self, texts):
        counts = Counter(w for t in texts for w in self._split(t))
        self.itos = ["[PAD]", "[UNK]", "[CLS]", "[SEP]"] + [w for w, _ in counts.most_common()]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_token_id = 0

    @staticmethod
    def _split(text):
        return re.findall(r"[a-z0-9']+|[^\sa-z0-9']", text.lower())

    def __len__(self):
        return len(self.itos)

    def __call__(self, texts, padding=True, truncation=True, max_length=64, return_tensors="pt", **kw):
        if isinstance(texts, str):
            texts = [texts]
        seqs = [[2] + [self.stoi.get(w, 1) for w in self._split(t)][:max_length - 2] + [3] for t in texts]
        L = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), L), self.pad_token_id, dtype=torch.long)
        mask = torch.zeros((len(seqs), L), dtype=torch.long)
        for i, s in enumerate(seqs):
            ids[i, :len(s)] = torch.tensor(s); mask[i, :len(s)] = 1
        return {"input_ids": ids, "attention_mask": mask}


class SentimentDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts, self.labels = list(texts), list(labels)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        return self.texts[i], self.labels[i]


def make_collate(tokenizer, max_length):
    def collate(batch):
        texts, labels = zip(*batch)
        enc = dict(tokenizer(list(texts), padding=True, truncation=True, max_length=max_length,
                             return_tensors="pt"))
        enc.pop("token_type_ids", None)          # segment ids: DistilBERT has none, single sentences need none
        enc["labels"] = torch.tensor(labels, dtype=torch.long)
        return enc
    return collate


cls_tok = bert_tok if bert_tok is not None else SimpleWordTokenizer(train_texts)
MAX_LEN = 64
BATCH_SIZE = 16 if FAST_RUN else 32              # smaller batches under FAST_RUN -> more optimizer steps per epoch
train_ds, val_ds = SentimentDataset(train_texts, train_labels), SentimentDataset(val_texts, val_labels)
collate = make_collate(cls_tok, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate,
                          num_workers=0, pin_memory=(device.type == "cuda"),
                          generator=torch.Generator().manual_seed(SEED))
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, collate_fn=collate, num_workers=0,
                        pin_memory=(device.type == "cuda"))

batch = next(iter(train_loader))
print("tokenizer:", type(cls_tok).__name__, "| batch keys:", list(batch.keys()))
print({k: tuple(v.shape) for k, v in batch.items()})

# how much padding does dynamic padding save compared with padding everything to MAX_LEN?
dyn = sum(b["input_ids"].numel() for b in train_loader)
fixed = sum(b["input_ids"].size(0) * MAX_LEN for b in train_loader)
real = sum(int(b["attention_mask"].sum()) for b in train_loader)
print(f"tokens processed per epoch: fixed padding {fixed:,} | dynamic padding {dyn:,} | real tokens {real:,}")

Dynamic padding cuts the processed tokens by a large factor for short sentences (sorting the data by
length before batching, as `Trainer`'s `group_by_length` option does, would shave off most of what
remains). Note that this only works because the attention mask tells the model which positions to
ignore; padding without a mask would silently change the predictions.

### 4.3 The fine-tuning loop, written by hand

`AutoModelForSequenceClassification` builds DistilBERT with a fresh classification head: a linear
layer on the `[CLS]` hidden state (with a `pre_classifier` projection and dropout in between). The
head is randomly initialized, which is why the library warns that some weights were "newly
initialized"; that is expected. If we pass `labels=...` to the forward call the model computes the
cross-entropy itself and returns it as `out.loss`, which keeps the training loop short.

The optimizer recipe is the one from the BERT paper and has barely changed since: **AdamW** with a
small learning rate ($2\times10^{-5}$ to $5\times10^{-5}$ for full fine-tuning), weight decay 0.01, a
**linear warmup** over the first ~10% of steps followed by a **linear decay** to zero, gradient
clipping at norm 1, and only 2–4 epochs. Warmup matters because the pretrained weights are in a
delicate region and large early updates destroy what was learned ("catastrophic forgetting"); the low
learning rate keeps the encoder close to its pretrained solution. `transformers` provides the schedule
as `get_linear_schedule_with_warmup`; we also write the equivalent `LambdaLR` ourselves so that the
loop works without the library and so that you can see that there is nothing magic inside.

In [ ]:
def linear_warmup_decay(optimizer, warmup_steps, total_steps):
    """LambdaLR that ramps linearly to the base LR over warmup_steps and then decays linearly to 0."""
    def factor(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, factor)


def train_one_epoch(model, loader, optimizer, scheduler=None, grad_clip=1.0):
    model.train()
    total, n = 0.0, 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        optimizer.zero_grad(set_to_none=True)
        out.loss.backward()
        if grad_clip:
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], grad_clip)
        optimizer.step()
        if scheduler is not None:
            scheduler.step()
        bs = batch["labels"].size(0)
        total += out.loss.item() * bs; n += bs
    return total / n


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, n, preds, targets = 0.0, 0, [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        bs = batch["labels"].size(0)
        total += out.loss.item() * bs; n += bs
        preds.append(out.logits.argmax(-1).cpu()); targets.append(batch["labels"].cpu())
    preds, targets = torch.cat(preds).cpu().numpy(), torch.cat(targets).cpu().numpy()
    return {"loss": total / n, "acc": accuracy_score(targets, preds), "f1": f1_score(targets, preds)}


def fit(model, train_loader, val_loader, epochs, lr, weight_decay=0.01, warmup_frac=0.1, tag=""):
    """AdamW + linear warmup/decay; prints one line per epoch and returns the history."""
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    total_steps = epochs * len(train_loader)
    warmup = int(warmup_frac * total_steps)
    if get_linear_schedule_with_warmup is not None:
        scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total_steps)
    else:
        scheduler = linear_warmup_decay(optimizer, warmup, total_steps)
    history = []
    t_start = time.time()
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, **val})
        print(f"{tag}epoch {epoch} | train loss {train_loss:.4f} | val loss {val['loss']:.4f} | "
              f"val acc {val['acc']:.3f} | val F1 {val['f1']:.3f} | {time.time() - t0:.1f}s")
    history[-1]["time"] = time.time() - t_start
    return history


# the two schedules agree exactly
_opt = torch.optim.SGD([nn.Parameter(torch.zeros(1))], lr=1.0)
_ours = linear_warmup_decay(_opt, 4, 20)
ours = []
for _ in range(20):
    ours.append(_opt.param_groups[0]["lr"]); _opt.step(); _ours.step()
if get_linear_schedule_with_warmup is not None:
    _opt2 = torch.optim.SGD([nn.Parameter(torch.zeros(1))], lr=1.0)
    _hf = get_linear_schedule_with_warmup(_opt2, 4, 20)
    theirs = []
    for _ in range(20):
        theirs.append(_opt2.param_groups[0]["lr"]); _opt2.step(); _hf.step()
    print("hand-written schedule == transformers schedule:", np.allclose(ours, theirs))
plt.plot(ours, marker="o"); plt.xlabel("step"); plt.ylabel("LR multiplier")
plt.title("Linear warmup (4 steps) then linear decay (20 steps total)"); plt.tight_layout(); plt.show()

Now the actual fine-tuning. `EPOCHS_FT` is one epoch under `FAST_RUN` (512 sentences in batches of
16, i.e. thirty-two optimizer steps, well under a minute on a laptop CPU) and three epochs on the 4k
subset in batches of 32 otherwise, which reaches about 88–90% validation accuracy on SST-2; the full
67k training set gets DistilBERT to ~91%. We keep a `results` dictionary so that Section 5 can compare
full fine-tuning against LoRA in one table.

In [ ]:
EPOCHS_FT = 1 if FAST_RUN else 3
LR_FULL, LR_LORA = 5e-5, 1e-3                  # full fine-tuning vs adapter learning rates (see Section 5)
results = {}

ft_model = hf_load(AutoModelForSequenceClassification, "distilbert-base-uncased", num_labels=2)
if ft_model is not None:
    ft_model = ft_model.to(device)
    total, trainable = count_params(ft_model)
    print(f"full fine-tuning: {trainable / 1e6:.2f}M trainable of {total / 1e6:.2f}M parameters")
    torch.manual_seed(SEED)
    hist = fit(ft_model, train_loader, val_loader, epochs=EPOCHS_FT, lr=LR_FULL, tag="[full FT] ")
    results["full fine-tuning"] = {"trainable": trainable, **hist[-1]}
else:
    skip("fine-tuning needs the DistilBERT weights")

How far the accuracy moves depends entirely on the budget. Thirty-two optimizer steps (the `FAST_RUN`
setting) are a smoke test of the pipeline: the loss starts to fall and the accuracy typically climbs
into the 60–80% range, but the run is far from converged, and with an unlucky seed it can still sit
near the 50% chance level after the first epoch, because the fresh classification head first has to
find the sentiment direction in the `[CLS]` features. With the full budget the encoder's pretrained
features carry the model to ~90% within the first epoch. Watch the validation loss in the multi-epoch
run: it typically bottoms out after two or three epochs and then creeps up while accuracy stays flat,
the usual sign that the model starts to memorize the training subset. Exercise 3 adds early stopping
to `fit`.

### 4.4 Inference on your own sentences

For deployment we want a function that takes raw strings and returns labels with probabilities. It is
the evaluation code without labels: tokenize with padding, forward under `torch.no_grad()`, softmax.

In [ ]:
@torch.no_grad()
def predict(model, tokenizer, texts, max_length=MAX_LEN):
    model.eval()
    enc = dict(tokenizer(list(texts), padding=True, truncation=True, max_length=max_length, return_tensors="pt"))
    enc.pop("token_type_ids", None)
    enc = {k: v.to(device) for k, v in enc.items()}
    probs = model(**enc).logits.softmax(-1)
    return probs.argmax(-1).cpu().tolist(), probs.max(-1).values.cpu().tolist()


custom = ["An absolute triumph, I could not stop smiling.",
          "Two hours of my life I will never get back.",
          "The plot is thin but the performances carry it.",
          "It is not bad, exactly, but it is not good either."]
if ft_model is not None:
    for text, label, p in zip(custom, *predict(ft_model, cls_tok, custom)):
        print(f"{'positive' if label else 'negative':>8s} ({p:.2f})  {text}")
else:
    skip("inference demo needs the fine-tuned model")

The last two sentences are deliberately mixed; look at the confidence values, not just the labels. A
model trained on a few hundred examples is usually overconfident on such inputs, which is one of the
themes of Notebook 16 · Interpretability, Robustness and Uncertainty.

### 4.5 The same thing with `Trainer`

The `transformers.Trainer` class packages the loop we just wrote, plus mixed precision, gradient
accumulation, logging, checkpointing, distributed training and evaluation callbacks, behind a
configuration object (`TrainingArguments`). You give it a model, tokenized datasets (any `Dataset`
whose items are dictionaries of tensors or lists), a collator that pads batches
(`DataCollatorWithPadding` does the dynamic padding for us) and a `compute_metrics` function. Because
the library's keyword arguments have been renamed across versions (`evaluation_strategy` became
`eval_strategy`, `tokenizer=` became `processing_class=`), the helper below inspects the installed
version's signature and adapts, a small example of writing code that survives API drift.

In [ ]:
class TokenizedDataset(Dataset):
    """Pre-tokenized examples as dicts (input_ids, attention_mask, labels): what Trainer expects."""

    def __init__(self, texts, labels, tokenizer, max_length):
        self.enc = tokenizer(list(texts), truncation=True, max_length=max_length)   # lists, no padding yet
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: self.enc[k][i] for k in ("input_ids", "attention_mask")}
        item["labels"] = self.labels[i]
        return item


def make_training_args(**kwargs):
    """Build TrainingArguments, renaming/dropping keys that this transformers version does not know."""
    params = inspect.signature(TrainingArguments.__init__).parameters
    if "eval_strategy" not in params and "evaluation_strategy" in params:
        kwargs["evaluation_strategy"] = kwargs.pop("eval_strategy")
    dropped = [k for k in kwargs if k not in params]
    for k in dropped:
        kwargs.pop(k)
    if dropped:
        print("ignored unsupported TrainingArguments:", dropped)
    return TrainingArguments(**kwargs)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.asarray(logits).argmax(-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}


trainer = None
tr_model = hf_load(AutoModelForSequenceClassification, "distilbert-base-uncased", num_labels=2)
if tr_model is not None:
    try:
        hf_train = TokenizedDataset(train_texts, train_labels, bert_tok, MAX_LEN)
        hf_val = TokenizedDataset(val_texts, val_labels, bert_tok, MAX_LEN)
        n_steps = EPOCHS_FT * math.ceil(len(hf_train) / BATCH_SIZE)
        args = make_training_args(
            output_dir=str(DATA_DIR / "hf_runs" / "sst2_trainer"), seed=SEED,
            per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=64,
            num_train_epochs=EPOCHS_FT, learning_rate=LR_FULL, weight_decay=0.01,
            warmup_steps=int(0.1 * n_steps),                  # same 10% linear warmup as our loop
            eval_strategy="epoch", save_strategy="no", logging_steps=5, report_to="none",
            use_cpu=(device.type != "cuda"), dataloader_num_workers=0, disable_tqdm=True)
        trainer_kwargs = dict(model=tr_model, args=args, train_dataset=hf_train, eval_dataset=hf_val,
                              data_collator=DataCollatorWithPadding(bert_tok), compute_metrics=compute_metrics)
        if "processing_class" in inspect.signature(Trainer.__init__).parameters:
            trainer_kwargs["processing_class"] = bert_tok
        else:
            trainer_kwargs["tokenizer"] = bert_tok
        trainer = Trainer(**trainer_kwargs)
        t0 = time.time()
        train_output = trainer.train()
        metrics = trainer.evaluate()
        print(f"Trainer: {time.time() - t0:.1f}s | train loss {train_output.training_loss:.4f} | "
              f"eval acc {metrics['eval_accuracy']:.3f} | eval F1 {metrics['eval_f1']:.3f}")
        results["Trainer (full FT)"] = {"trainable": count_params(tr_model)[1], "acc": metrics["eval_accuracy"],
                                        "f1": metrics["eval_f1"], "time": time.time() - t0}
    except Exception as e:
        skip(f"Trainer run failed ({type(e).__name__}: {str(e)[:120]}); it needs the `accelerate` package")
else:
    skip("the Trainer demo needs the DistilBERT weights")

The `Trainer` keeps a log of every logging step in `trainer.state.log_history`; it is the raw material
for the loss curves that the higher-level integrations (TensorBoard, Weights & Biases) would draw for
you. Plotting it ourselves is a good habit because it forces you to look at the numbers.

In [ ]:
if trainer is not None:
    logs = [r for r in trainer.state.log_history if "loss" in r and "step" in r]
    if logs:
        plt.plot([r["step"] for r in logs], [r["loss"] for r in logs], marker=".")
        plt.xlabel("optimizer step"); plt.ylabel("training loss"); plt.title("Trainer training loss")
        plt.tight_layout(); plt.show()
    evals = [r for r in trainer.state.log_history if "eval_accuracy" in r]
    for r in evals[:EPOCHS_FT]:                     # the per-epoch evaluations (the last entry is our explicit call)
        print(f"epoch {r.get('epoch', 0):.0f}: eval loss {r['eval_loss']:.4f}, accuracy {r['eval_accuracy']:.3f}")
    tr_model = trainer = None
    gc.collect()
else:
    skip("no Trainer run to plot")

**When to use which.** The hand-written loop is about forty lines and you control every one of them:
it is the right tool when you need an unusual loss, several models interacting (distillation,
adversarial training, DPO), non-standard batching, or when you want to understand what is going on.
`Trainer` is the right tool for standard supervised fine-tuning at scale: it gives you mixed precision,
gradient accumulation, checkpoint resumption, multi-GPU and logging integrations by flipping arguments,
and it is what most published fine-tuning recipes are written against. Both should reach the same
accuracy on the same data with the same hyperparameters once the budget is large enough to converge,
which is a useful sanity check when you write your own loop; after thirty-two steps (the `FAST_RUN`
budget) the two numbers differ by seed and data-order noise, which is itself a lesson about comparing
runs at tiny budgets (Notebook 17 · Capstone Projects returns to this point).

> **Common pitfall.** Forgetting that `Trainer` moves the model to the GPU and back, changes
> `model.train()`/`model.eval()` state, and by default *saves checkpoints every 500 steps* into
> `output_dir`. Set `save_strategy` deliberately, or your disk fills with 250 MB files.

Notebook 10 - part 3 of 4 (LoRA from scratch and via peft, how LLMs are trained, practical considerations)

## 5. Parameter-efficient fine-tuning: LoRA from scratch and with `peft`

### 5.1 Why not just fine-tune everything? The memory arithmetic

Full fine-tuning updates every weight, and every updated weight costs far more than its own storage.
With AdamW in 32-bit precision each parameter needs 4 bytes for the weight, 4 bytes for its gradient
and 8 bytes for the two Adam moment estimates, **16 bytes per parameter** before we count
activations. For DistilBERT that is about 1 GB, trivial. For a 7-billion-parameter model it is 112 GB,
which does not fit on any single GPU, and for every task you want to support you must store a complete
7B-parameter copy (14 GB in bfloat16). **Parameter-efficient fine-tuning** (PEFT) methods freeze the
pretrained weights and train a small number of *additional* parameters. The gradient and optimizer
memory shrink to almost nothing, the frozen weights can be stored in low precision, and each task's
adapter is a few megabytes that can be swapped in at runtime. The table below does the arithmetic.

In [ ]:
def finetune_memory_gb(n_params, n_trainable, weight_bytes=4, state_bytes=4):
    """(weights, gradients, Adam states, total) in GB, ignoring activations."""
    weights = n_params * weight_bytes
    grads = n_trainable * state_bytes
    adam = 2 * n_trainable * state_bytes
    return weights / 1e9, grads / 1e9, adam / 1e9, (weights + grads + adam) / 1e9


configs = [("DistilBERT 66M, full FT (fp32)", 66e6, 66e6, 4),
           ("DistilBERT 66M, LoRA r=8 (fp32)", 66e6, 0.74e6, 4),
           ("7B model, full FT (fp32)", 7e9, 7e9, 4),
           ("7B model, full FT (bf16 weights)", 7e9, 7e9, 2),
           ("7B model, LoRA r=16 (bf16 weights)", 7e9, 40e6, 2),
           ("7B model, QLoRA r=16 (4-bit weights)", 7e9, 40e6, 0.5)]
print(f"{'configuration':40s} {'weights':>8s} {'grads':>7s} {'Adam':>7s} {'total':>8s}   (GB)")
for name, n, n_tr, wb in configs:
    w, g, a, t = finetune_memory_gb(n, n_tr, weight_bytes=wb)
    print(f"{name:40s} {w:8.2f} {g:7.2f} {a:7.2f} {t:8.2f}")

### 5.2 LoRA: low-rank adaptation, derived

**LoRA** (Hu et al., 2021) starts from an empirical observation: the *change* in weights during
fine-tuning, $\Delta W = W_{\text{fine-tuned}} - W_{\text{pretrained}}$, has low intrinsic rank. If
that is so, we should not spend $d_{\text{out}} \times d_{\text{in}}$ parameters on it. LoRA
constrains the update to a product of two thin matrices,

$$
W' = W + \Delta W, \qquad \Delta W = \frac{\alpha}{r}\, B A,
\qquad A \in \mathbb{R}^{r \times d_{\text{in}}},\; B \in \mathbb{R}^{d_{\text{out}} \times r},
$$

with rank $r \ll \min(d_{\text{in}}, d_{\text{out}})$ (typically 4 to 64) and a scalar
$\alpha$ that sets the effective step size (the ratio $\alpha / r$ keeps the update magnitude roughly
constant when you change $r$). The frozen $W$ is never modified during training; the forward pass of
the adapted linear layer is

$$
h = W x + \frac{\alpha}{r}\, B (A x),
$$

which is computed as two thin matrix products rather than by forming $BA$ explicitly, so the extra
cost is $r(d_{\text{in}} + d_{\text{out}})$ multiply-adds per token instead of
$d_{\text{in}} d_{\text{out}}$. The parameter count is the same expression: for a $768 \times 768$
projection and $r = 8$, LoRA trains $8 \times (768 + 768) = 12{,}288$ parameters instead of
$589{,}824$, a 48-fold reduction.

Two initialization choices make the method work. $B$ is initialized to **zero** and $A$ randomly
(Kaiming-uniform), so that at the start of training $\Delta W = 0$ and the adapted model is *exactly*
the pretrained model; training then moves away from that point smoothly. And because $\Delta W$ is an
ordinary matrix, after training we can **merge** it into the frozen weight,
$W_{\text{merged}} = W + \frac{\alpha}{r} BA$, and serve the model with zero inference overhead (or
un-merge to swap in a different adapter). LoRA is usually applied to the attention projections
(query and value at minimum), sometimes to all linear layers; a higher learning rate than for full
fine-tuning ($10^{-4}$ to $10^{-3}$) is standard because the adapter starts from zero.

The implementation below wraps an existing `nn.Linear`, freezes it, adds `lora_A` and `lora_B`, and
provides `merge()` / `unmerge()`.

In [ ]:
class LoRALinear(nn.Module):
    """Wrap a frozen nn.Linear with a trainable low-rank update  W x + (alpha/r) * B A x."""

    def __init__(self, base: nn.Linear, r: int = 8, alpha: float = 16.0, dropout: float = 0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.r, self.alpha, self.scaling = r, alpha, alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))       # zero -> starts as identity update
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        self.merged = False

    def delta_weight(self):
        return self.scaling * (self.lora_B @ self.lora_A)                    # (out, in)

    def forward(self, x):
        out = self.base(x)
        if self.merged:
            return out
        return out + self.scaling * ((self.dropout(x) @ self.lora_A.t()) @ self.lora_B.t())

    @torch.no_grad()
    def merge(self):
        if not self.merged:
            self.base.weight += self.delta_weight()
            self.merged = True

    @torch.no_grad()
    def unmerge(self):
        if self.merged:
            self.base.weight -= self.delta_weight()
            self.merged = False

    def extra_repr(self):
        return f"r={self.r}, alpha={self.alpha}, merged={self.merged}"


def apply_lora(model, targets=("q_lin", "v_lin"), r=8, alpha=16.0, dropout=0.0):
    """Replace every nn.Linear whose name ends with one of `targets` by a LoRALinear. Returns the count."""
    replaced = 0
    for name, module in list(model.named_modules()):
        if isinstance(module, nn.Linear) and name.split(".")[-1] in targets:
            parent = model.get_submodule(name.rsplit(".", 1)[0]) if "." in name else model
            setattr(parent, name.split(".")[-1], LoRALinear(module, r=r, alpha=alpha, dropout=dropout))
            replaced += 1
    return replaced


def mark_only_lora_trainable(model, also_train=()):
    """Freeze everything except LoRA matrices and the modules named in `also_train` (e.g. the head)."""
    for name, p in model.named_parameters():
        parts = name.split(".")
        p.requires_grad_("lora_A" in parts or "lora_B" in parts or any(h in parts for h in also_train))


def lora_modules(model):
    return [m for m in model.modules() if isinstance(m, LoRALinear)]


# unit tests on a random linear layer
torch.manual_seed(SEED)
base = nn.Linear(64, 32)
x = torch.randn(5, 64)
lora = LoRALinear(base, r=4, alpha=8)
print("at init, LoRA output == base output:", torch.allclose(lora(x), base(x)))
with torch.no_grad():
    lora.lora_B.normal_(std=0.1)                    # pretend we trained the adapter
y_unmerged = lora(x)
lora.merge();   y_merged = lora(x)
lora.unmerge(); y_back = lora(x)
print("merged == unmerged:", torch.allclose(y_unmerged, y_merged, atol=1e-6),
      "| unmerge restores:", torch.allclose(y_unmerged, y_back, atol=1e-6))
loss = lora(x).pow(2).mean(); loss.backward()
print("grad on base weight:", base.weight.grad, "| grad on lora_A:", tuple(lora.lora_A.grad.shape),
      "| grad on lora_B:", tuple(lora.lora_B.grad.shape))
total, trainable = count_params(lora)
print(f"parameters: {trainable} trainable of {total} ({trainable / total:.1%}); "
      f"formula r*(in+out) = {4 * (64 + 32)}")

The tests confirm the three properties we derived: the wrapped layer is initially identical to the base
layer, merging is exact and reversible, and gradients reach only the adapter (the base weight's `.grad`
stays `None`). Now we apply it to DistilBERT's query and value projections, freeze everything but the
adapters and the classification head (the head is new and *must* be trained), and fine-tune with a
higher learning rate than in Section 4.

In [ ]:
LORA_R, LORA_ALPHA = 8, 16.0
lora_model = hf_load(AutoModelForSequenceClassification, "distilbert-base-uncased", num_labels=2)
if lora_model is not None:
    n_replaced = apply_lora(lora_model, targets=("q_lin", "v_lin"), r=LORA_R, alpha=LORA_ALPHA, dropout=0.05)
    mark_only_lora_trainable(lora_model, also_train=("pre_classifier", "classifier"))
    lora_model = lora_model.to(device)
    total, trainable = count_params(lora_model)
    n_lora = sum(m.lora_A.numel() + m.lora_B.numel() for m in lora_modules(lora_model))
    print(f"replaced {n_replaced} linear layers; trainable {trainable:,} of {total:,} "
          f"({trainable / total:.2%}), of which LoRA {n_lora:,} and head {trainable - n_lora:,}")
    print(lora_model.distilbert.transformer.layer[0].attention.q_lin)
    torch.manual_seed(SEED)
    hist = fit(lora_model, train_loader, val_loader, epochs=EPOCHS_FT, lr=LR_LORA, tag="[LoRA scratch] ")
    results["LoRA from scratch"] = {"trainable": trainable, **hist[-1]}
else:
    skip("the LoRA fine-tuning needs the DistilBERT weights")

With about 1% of the parameters trainable, LoRA reaches an accuracy in the same range as full
fine-tuning on this task (often within a point or two, sometimes better on tiny training sets because
the constrained update regularizes). The training step is also cheaper: no gradients flow into the
frozen weights' optimizer states, and the backward pass through the adapters is small, although the
activations of the full network are still needed, so LoRA is not free.

Before serving we merge the adapters. The predictions must not change, which the next cell verifies on
a validation batch, and the merged model is once again a plain DistilBERT that any inference code can
load.

In [ ]:
if lora_model is not None:
    batch = {k: v.to(device) for k, v in next(iter(val_loader)).items()}
    lora_model.eval()
    with torch.no_grad():
        logits_before = lora_model(**batch).logits
        for m in lora_modules(lora_model):
            m.merge()
        logits_after = lora_model(**batch).logits
        for m in lora_modules(lora_model):
            m.unmerge()
    print("max |logit difference| after merging:", (logits_before - logits_after).abs().max().item())
    print("merged predictions identical:", torch.equal(logits_before.argmax(-1), logits_after.argmax(-1)))
    # what an adapter costs on disk: only the LoRA matrices need to be saved
    adapter_state = {k: v for k, v in lora_model.state_dict().items() if "lora_" in k}
    n_bytes = sum(v.numel() * v.element_size() for v in adapter_state.values())
    print(f"adapter checkpoint: {len(adapter_state)} tensors, {n_bytes / 1e6:.2f} MB "
          f"(full model: {sum(p.numel() * p.element_size() for p in lora_model.parameters()) / 1e6:.0f} MB)")
else:
    skip("merging demo needs the LoRA model")

### 5.3 The `peft` library

Hugging Face's `peft` library implements LoRA (and a dozen relatives) for any `transformers` model.
You describe the adapter in a `LoraConfig`, call `get_peft_model`, and get back a wrapper whose forward
pass and `state_dict` handle everything we wrote by hand: injecting `LoRALinear`-like modules into the
named target modules, freezing the rest, keeping the task head trainable (`task_type=SEQ_CLS` marks
the classifier as a module to save), saving only the adapter weights with `save_pretrained`, and
merging with `merge_and_unload()`. The training loop is unchanged: a peft model is still an
`nn.Module` that returns `out.loss` when given labels.

In [ ]:
peft_model = None
if PEFT_AVAILABLE and HF_AVAILABLE:
    base_model = hf_load(AutoModelForSequenceClassification, "distilbert-base-uncased", num_labels=2)
    if base_model is not None:
        try:
            lora_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=LORA_R, lora_alpha=LORA_ALPHA,
                                     lora_dropout=0.05, target_modules=["q_lin", "v_lin"],
                                     modules_to_save=["pre_classifier", "classifier"])   # the new head stays trainable
            peft_model = get_peft_model(base_model, lora_config).to(device)
            peft_model.print_trainable_parameters()
            torch.manual_seed(SEED)
            hist = fit(peft_model, train_loader, val_loader, epochs=EPOCHS_FT, lr=LR_LORA, tag="[peft LoRA] ")
            results["LoRA via peft"] = {"trainable": count_params(peft_model)[1], **hist[-1]}
        except Exception as e:
            skip(f"peft run failed: {type(e).__name__}: {str(e)[:120]}")
            peft_model = None
else:
    skip("the peft demo needs both `transformers` and `peft` (see environment.yml)")

The trainable-parameter count reported by `peft` matches our from-scratch version (the LoRA matrices
have the same shapes, and `modules_to_save` keeps a trainable copy of the head, just as our `also_train`
argument did), and so does the accuracy: the two implementations are the same algorithm. In practice you would now call `peft_model.save_pretrained(path)` to write an adapter of a
few megabytes, and `merge_and_unload()` to obtain a merged plain model for deployment. Let us compare
all the fine-tuning runs of this notebook.

In [ ]:
if results:
    print(f"{'method':22s} {'trainable':>12s} {'val acc':>8s} {'val F1':>7s} {'time (s)':>9s}")
    for name, r in results.items():
        print(f"{name:22s} {r['trainable']:12,d} {r['acc']:8.3f} {r['f1']:7.3f} {r.get('time', float('nan')):9.1f}")
    names = list(results)
    fig, ax1 = plt.subplots(figsize=(7, 3.8))
    ax1.bar(names, [results[n]["acc"] for n in names], color="C0", alpha=0.8)
    ax1.set_ylabel("validation accuracy"); ax1.set_ylim(0.4, 1.0)   # a FAST_RUN smoke test can sit near chance
    for i, n in enumerate(names):
        ax1.text(i, results[n]["acc"] + 0.01, f"{results[n]['trainable'] / 1e6:.2f}M params", ha="center", fontsize=8)
    ax1.set_title("Fine-tuning methods: accuracy and trainable parameters")
    plt.xticks(rotation=10); plt.tight_layout(); plt.show()
else:
    skip("no fine-tuning results to compare (pretrained weights were unavailable)")
if peft_model is not None:
    try:
        merged = peft_model.merge_and_unload()
        print("merge_and_unload ->", type(merged).__name__)
    except Exception as e:
        print("merge_and_unload failed:", type(e).__name__)
peft_model = None
gc.collect()

### 5.4 The wider PEFT family

LoRA is the most popular PEFT method, but it is one point in a design space.

- **Adapters** (Houlsby et al., 2019) insert small bottleneck MLPs (down-projection, nonlinearity,
  up-projection, residual) after the attention and feed-forward sub-layers. They work well but, unlike
  LoRA, cannot be merged away, so they add inference latency.
- **Prefix tuning** and **prompt tuning** (Li and Liang, 2021; Lester et al., 2021) leave the whole
  network frozen and learn a handful of *virtual tokens* whose keys and values (prefix tuning) or
  input embeddings (prompt tuning) are prepended to every sequence. They train even fewer parameters
  than LoRA, and prompt tuning becomes competitive only for very large models.
- **IA3** (Liu et al., 2022) learns per-channel scaling vectors for keys, values and feed-forward
  activations, roughly ten times fewer parameters than LoRA (Exercise 5 implements it or DoRA).
- **QLoRA** (Dettmers et al., 2023) combines LoRA with a 4-bit quantized frozen base model (the NF4
  data type plus double quantization) and paged optimizers. Fine-tuning a 65B-parameter model on a
  single 48 GB GPU became possible this way; the last row of the memory table in Section 5.1 is
  QLoRA's regime.
- **DoRA** (Liu et al., 2024) decomposes each weight into a magnitude vector and a direction, applies
  LoRA to the direction only, and trains the magnitude separately; it closes most of the remaining gap
  to full fine-tuning at the same parameter budget.

> **Key idea.** Freeze the pretrained weights and train a small, structured update. LoRA's low-rank
> update is cheap to train, cheap to store and free at inference time after merging.

## 6. How large language models are trained today

The models we fine-tuned are tiny by today's standards, but the recipe that produces a modern
assistant model has the same ingredients we have been using, stacked in stages. This section describes
those stages in prose and implements the two loss functions that are new: the masked SFT loss and the
DPO loss.

### 6.1 Pretraining

Stage one is exactly the causal language modelling of Notebook 09 · Attention and Transformers, scaled
up: a decoder-only
Transformer with $10^9$ to $10^{12}$ parameters is trained to predict the next token on a corpus of
$10^{12}$ to $10^{13}$ tokens of web text, books, code and curated data, using AdamW, a cosine
schedule with warmup, and thousands of accelerators for weeks. The scaling laws discussed in Notebook 09
and Notebook 14 · Efficient Training and Scaling decide how to split a compute budget between
parameters and tokens; the data mixture
(how much code, how much of which languages, what quality filters and deduplication) is at least as
important as the architecture. The result, a **base model**, is a superb text predictor and a poor
assistant: asked a question it may continue with more questions, because that is what text on the
internet does.

### 6.2 Supervised fine-tuning (SFT)

Stage two teaches the format of a conversation. The base model is fine-tuned on tens of thousands of
(prompt, ideal response) pairs written by humans or synthesized by stronger models. Mechanically this is
still next-token prediction with the same cross-entropy loss, with one important detail: the loss is
computed **only on the response tokens**. The prompt is given as context, but the model is not
rewarded for predicting it. In PyTorch this is implemented by setting the label of every prompt
position to the `ignore_index` (-100 by convention), which `F.cross_entropy` skips. The cell below
shows that the masked loss equals the average over response positions only.

In [ ]:
V = 50                                                 # toy vocabulary
prompt_ids = torch.tensor([11, 24, 7, 3])              # "user: what is LoRA?" (pretend)
response_ids = torch.tensor([19, 42, 8, 2])            # "a low-rank update <eos>" (pretend)
input_ids = torch.cat([prompt_ids, response_ids]).unsqueeze(0)          # (1, T)
labels = input_ids.clone()
labels[:, :len(prompt_ids)] = -100                     # no loss on prompt positions
torch.manual_seed(SEED)
logits = torch.randn(1, input_ids.size(1), V)          # stand-in for the model output

shift_logits, shift_labels = logits[:, :-1], labels[:, 1:]              # predict token t+1 from position t
masked_loss = F.cross_entropy(shift_logits.reshape(-1, V), shift_labels.reshape(-1), ignore_index=-100)
per_token = F.cross_entropy(shift_logits.reshape(-1, V), input_ids[:, 1:].reshape(-1), reduction="none")
manual = per_token[shift_labels.reshape(-1) != -100].mean()
print("labels after masking:", labels[0].tolist())
print(f"masked loss {masked_loss:.4f} == mean over response positions {manual:.4f}:",
      torch.allclose(masked_loss, manual))

### 6.3 Preference optimization: from RLHF to DPO

An SFT model imitates its demonstrations, including their flaws, and it has no notion of which of two
acceptable answers is *better*. Stage three fixes this with **human preference data**: annotators see
a prompt $x$ and two model responses and pick the preferred one, giving triples $(x, y_w, y_l)$
("winner" and "loser"). The classic pipeline, **RLHF** (Christiano et al., 2017; Ouyang et al., 2022,
InstructGPT), first fits a **reward model** $r_\phi(x, y)$ to these comparisons with the
Bradley–Terry model of pairwise preferences,

$$
p(y_w \succ y_l \mid x) = \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big),
$$

and then optimizes the language model $\pi_\theta$ with reinforcement learning (PPO, see Notebook 13 ·
Reinforcement Learning) to maximize the reward while staying close to the SFT model $\pi_{\text{ref}}$:

$$
\max_{\pi_\theta}\; \mathbb{E}_{x,\, y \sim \pi_\theta(\cdot|x)}\big[r_\phi(x, y)\big]
- \beta\, \mathrm{KL}\big(\pi_\theta(\cdot|x)\,\|\,\pi_{\text{ref}}(\cdot|x)\big).
$$

The KL penalty with coefficient $\beta$ prevents the policy from drifting to degenerate text that
exploits the reward model. This works but is complex: four models in memory (policy, reference, reward,
value), on-policy sampling, and PPO's many hyperparameters.

**Direct Preference Optimization** (Rafailov et al., 2023) removes the reward model and the RL loop
with a short derivation. The KL-regularized objective above has a closed-form optimum,

$$
\pi^*(y \mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y \mid x)\, \exp\!\big(r(x, y) / \beta\big),
$$

which we can solve for the reward: $r(x, y) = \beta \log \frac{\pi^*(y|x)}{\pi_{\text{ref}}(y|x)} +
\beta \log Z(x)$. Substituting this into the Bradley–Terry likelihood makes the intractable
normalizer $Z(x)$ cancel, because both responses share the same prompt, and leaves a loss that depends
only on the policy and the reference:

$$
\mathcal{L}_{\text{DPO}}(\theta) = -\,\mathbb{E}_{(x, y_w, y_l)}\left[\log \sigma\!\left(
\beta \log \frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} -
\beta \log \frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right)\right].
$$

The quantity $\beta \log \frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$ is called the **implicit
reward**; the loss is a logistic regression that asks the implicit reward of the winner to exceed that
of the loser. Training needs only the log-probabilities of the two responses under the policy and
under the frozen reference model, which is one forward pass each per response, no sampling. $\beta$
plays the same role as before: small $\beta$ lets the policy move far from the reference, large
$\beta$ keeps it close.

In [ ]:
def dpo_loss(pi_chosen_logp, pi_rejected_logp, ref_chosen_logp, ref_rejected_logp, beta=0.1):
    """DPO loss for a batch of preference pairs. Returns (loss, chosen_rewards, rejected_rewards)."""
    chosen_rewards = beta * (pi_chosen_logp - ref_chosen_logp)        # implicit rewards
    rejected_rewards = beta * (pi_rejected_logp - ref_rejected_logp)
    margin = chosen_rewards - rejected_rewards
    loss = -F.logsigmoid(margin).mean()
    return loss, chosen_rewards.detach(), rejected_rewards.detach()


# a toy world: 4 prompts, 6 candidate responses each, a tabular policy over the responses
n_prompts, n_resp = 4, 6
torch.manual_seed(SEED); random.seed(SEED)
ref_logits = torch.randn(n_prompts, n_resp)                       # the frozen reference (SFT) policy
true_reward = 2.0 * torch.randn(n_prompts, n_resp)                # hidden "human" reward, never seen by DPO
pairs = []                                                        # (prompt, chosen, rejected)
for p in range(n_prompts):
    for _ in range(12):
        i, j = random.sample(range(n_resp), 2)
        p_i = torch.sigmoid(true_reward[p, i] - true_reward[p, j]).item()   # Bradley-Terry annotator
        pairs.append((p, i, j) if random.random() < p_i else (p, j, i))
P = torch.tensor([p for p, _, _ in pairs]); C = torch.tensor([c for _, c, _ in pairs]); R = torch.tensor([r for _, _, r in pairs])
ref_logp = ref_logits.log_softmax(-1)
print(f"{len(pairs)} preference pairs; reference agrees with the annotator on "
      f"{(ref_logp[P, C] > ref_logp[P, R]).float().mean():.0%} of them")


def train_dpo(beta, steps=150, lr=0.1):
    theta = nn.Parameter(ref_logits.clone())                      # policy initialized at the reference
    opt = torch.optim.Adam([theta], lr=lr)
    log = {"loss": [], "margin": [], "kl": [], "acc": []}
    for _ in range(steps):
        pi_logp = theta.log_softmax(-1)
        loss, r_c, r_r = dpo_loss(pi_logp[P, C], pi_logp[P, R], ref_logp[P, C], ref_logp[P, R], beta)
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            kl = (pi_logp.exp() * (pi_logp - ref_logp)).sum(-1).mean()       # KL(pi || ref), averaged over prompts
        log["loss"].append(loss.item()); log["margin"].append((r_c - r_r).mean().item())
        log["kl"].append(kl.item()); log["acc"].append((r_c > r_r).float().mean().item())
    return theta.detach(), log


runs = {beta: train_dpo(beta) for beta in [0.1, 0.5, 2.0]}
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
for beta, (_, log) in runs.items():
    axes[0].plot(log["loss"], label=f"beta={beta}")
    axes[1].plot(log["acc"], label=f"beta={beta}")
    axes[2].plot(log["kl"], label=f"beta={beta}")
axes[0].set_title("DPO loss"); axes[1].set_title("implicit-reward accuracy"); axes[2].set_title("KL(policy || reference)")
for ax in axes:
    ax.set_xlabel("step"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# does the learned policy agree with the hidden reward that DPO never saw?
for beta, (theta, log) in runs.items():
    implicit = beta * (theta.log_softmax(-1) - ref_logp)             # implicit reward for every (prompt, response)
    # rank agreement with the hidden true reward over all response pairs of each prompt
    agree, n = 0, 0
    for p in range(n_prompts):
        for i in range(n_resp):
            for j in range(i + 1, n_resp):
                agree += int((implicit[p, i] > implicit[p, j]) == (true_reward[p, i] > true_reward[p, j])); n += 1
    print(f"beta={beta:<4}: final loss {log['loss'][-1]:.3f}, KL {log['kl'][-1]:.2f}, "
          f"implicit reward agrees with the hidden reward on {agree / n:.0%} of all pairs")

Read the three panels together. The loss falls and the **implicit-reward accuracy**, the fraction of
training pairs on which the policy's implicit reward ranks the winner above the loser, climbs towards
100% for every $\beta$: a tabular policy has one free logit per response, so it can satisfy any
*consistent* set of preferences exactly, and the only pairs it cannot fit are contradictory ones (the
noisy Bradley–Terry annotator may label the same pair twice with opposite outcomes, or produce a
cycle). The difference between the runs is *how far the policy moves*: with $\beta = 0.1$ the KL to the
reference keeps growing throughout training as the policy sharpens onto the preferred responses, with
$\beta = 2$ it stays small. That is the regularization knob that keeps a real model from forgetting its
language ability while learning preferences; typical values for LLMs are $0.05$ to $0.5$. The printed
lines ask whether the learned policy ranks responses in agreement with the *hidden* true reward, which
DPO never saw directly, using the implicit reward as a stand-in reward model: the agreement is well
above chance for every $\beta$ (the 48 noisy comparisons carry real information about the 60 underlying
pairs), and what $\beta$ changes is the KL price paid for it.

### 6.4 Instruction following, evaluation and safety

The three stages produce a model that follows instructions, but a deployed assistant needs more.
**Chat templates** define how system prompts, user turns and assistant turns are serialized into a
token sequence (with special tokens such as `<|im_start|>`); tokenizers ship the template of their
model as `tokenizer.apply_chat_template`, and using the wrong template silently degrades quality.
**Evaluation** uses a battery of benchmarks (knowledge, reasoning, code, instruction following),
human preference arenas, and increasingly "LLM-as-a-judge" protocols; every one of them has failure
modes, from benchmark contamination (test questions leaking into the pretraining set, see Section 7)
to judges that prefer longer answers. **Safety** work spans the whole pipeline: data filtering during
pretraining, refusal and helpfulness demonstrations in SFT, preference data that penalizes harmful
completions, red-teaming, and post-hoc classifiers. Preference optimization is where much of a model's
"personality" is set, which is why the DPO loss you just implemented matters far beyond its ten lines.

The optional cell below runs a small instruction-tuned model (about 1 GB of weights) through its chat
template. It is skipped under `FAST_RUN` and whenever the download is impossible.

In [ ]:
INSTRUCT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
if FAST_RUN:
    skip(f"optional instruct-model demo ({INSTRUCT_MODEL}, ~1 GB download) is disabled under FAST_RUN")
elif not HF_AVAILABLE:
    skip("optional instruct-model demo needs transformers")
else:
    tok_i = hf_load(AutoTokenizer, INSTRUCT_MODEL)
    model_i = hf_load(AutoModelForCausalLM, INSTRUCT_MODEL)
    if tok_i is not None and model_i is not None:
        try:
            model_i = model_i.to(device).eval()
            messages = [{"role": "system", "content": "You are a concise teaching assistant."},
                        {"role": "user", "content": "Explain LoRA fine-tuning in two sentences."}]
            chat = tok_i.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            print("--- templated prompt ---\n" + chat)
            enc = tok_i(chat, return_tensors="pt").to(device)
            with torch.no_grad():
                out = model_i.generate(**enc, max_new_tokens=80, do_sample=False)
            print("--- answer ---\n" + tok_i.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))
        except Exception as e:
            skip(f"instruct-model generation failed: {type(e).__name__}: {str(e)[:120]}")
        model_i = None
        gc.collect()

## 7. Practical considerations

### 7.1 Sequence length and memory

Attention builds a `(batch, heads, L, L)` score matrix per layer. In fp16 with 12 heads, a batch of 8
and 6 layers, the score matrices alone take $8 \times 12 \times L^2 \times 2$ bytes per layer, which is
negligible at $L = 128$ and catastrophic at $L = 32{,}768$. The table below evaluates the formula.
FlashAttention-style kernels (`F.scaled_dot_product_attention` picks them automatically on supported
GPUs) never materialize the matrix and make memory *linear* in $L$, but the compute is still quadratic,
and the activations of the feed-forward layers still scale with $L$. The practical rules: truncate to
the shortest `max_length` that covers your data (inspect the token-length distribution), use dynamic
padding and length-grouped batching, and trade batch size for sequence length rather than the other
way round.

### 7.2 Mixed precision

Pretrained Transformers are routinely run and fine-tuned in 16-bit precision. On a GPU,
`torch.amp.autocast` with `float16` (plus a `GradScaler` for training) or `bfloat16` (no scaler needed
thanks to its wider exponent range) halves activation memory and doubles throughput on tensor cores;
on recent CPUs `bfloat16` autocast is also supported. Notebook 14 · Efficient Training and Scaling
covers the details; the second half of the next cell only checks that a forward pass under autocast
agrees with full precision to within the expected rounding error (a difference of the order of
$10^{-2}$ on hidden states of order 1 is normal for `bfloat16`, which keeps only 7 explicit mantissa
bits, about three significant decimal digits, and lets the rounding errors of six layers accumulate).

In [ ]:
B, H, LAYERS, BYTES = 8, 12, 6, 2
print(f"{'L':>7s} {'scores per layer':>17s} {'all layers':>12s}")
for L in [128, 512, 2048, 8192, 32768]:
    per_layer = B * H * L * L * BYTES
    print(f"{L:7d} {per_layer / 1e9:14.3f} GB {per_layer * LAYERS / 1e9:9.2f} GB")
if bert_tok is not None:
    lengths = [len(bert_tok(t)["input_ids"]) for t in train_texts]
    print(f"\ntoken lengths of the training sentences: mean {np.mean(lengths):.1f}, "
          f"95th percentile {np.percentile(lengths, 95):.0f}, max {max(lengths)} (MAX_LEN = {MAX_LEN})")

# mixed precision: does a forward pass under autocast agree with fp32?
if bert is not None and bert_tok is not None:
    amp_dtype = torch.float16 if device.type == "cuda" else torch.bfloat16
    enc = bert_tok(corpus[:4], padding=True, return_tensors="pt").to(device)
    try:
        with torch.no_grad():
            h32 = bert(**enc).last_hidden_state
            with torch.amp.autocast(device_type=device.type, dtype=amp_dtype):
                h16 = bert(**enc).last_hidden_state
        print(f"autocast dtype {h16.dtype}; max |difference| vs fp32: {(h32 - h16.float()).abs().max().item():.4f} "
              f"(hidden-state scale ~{h32.abs().mean().item():.2f})")
    except Exception as e:
        skip(f"autocast is not supported here: {type(e).__name__}: {str(e)[:80]}")
else:
    skip("mixed-precision check needs the DistilBERT weights")

### 7.3 Model cards, licenses and data contamination

Three non-technical issues decide whether a pretrained model may be used at all.

**Model cards** (Mitchell et al., 2019) are the README of a checkpoint on the Hub: intended use,
training data, evaluation results, known limitations and biases. Read them; DistilBERT's card, for
instance, documents that the model reproduces gender stereotypes present in its training data
(fill-mask on "The man worked as a [MASK]" versus "The woman worked as a [MASK]" makes this visible in
one line). Notebook 15 · Deployment and Model Optimization has you write a model card of your own.

**Licenses** vary. Many checkpoints are Apache-2.0 or MIT and can be used commercially; others are
released under "community" licenses with usage restrictions or acceptable-use policies, some are
research-only, and gated models require accepting terms before download. The license of the *training
data* is a separate question that the model license does not settle. Record the license of every
model and dataset in your project.

**Data contamination** means that a model's pretraining corpus contained the benchmark you evaluate it
on. With web-scale corpora this is the norm rather than the exception, and it inflates reported
results, sometimes dramatically. For your own projects the practical defence is to evaluate on data that
demonstrably post-dates the model's training cut-off or that you collected yourself, to report
contamination checks (n-gram overlap between your test set and public corpora) when you can, and to
be suspicious of a small model that scores far above its weight class on a famous benchmark.

> **Common pitfall.** Fine-tuning on a dataset whose test split is also on the Hub. If you evaluate
> on SST-2's public validation set, as we did, your numbers are comparable to the literature only
> because the labels of the *test* split were never released; never tune hyperparameters on the split
> you report.

Notebook 10 - part 4 of 4 (exercises, summary, further reading)

## Exercises

### Exercise 1 (warm-up): BPE decoding edge cases

Our `bpe_encode`/`bpe_decode` pair from Section 1.2 has three rough edges: it never maps tokens to
integer ids, characters that were not in the training alphabet (accents, digits, emoji) pass through as
symbols that no vocabulary entry knows about, and the decoder puts a space before every punctuation
mark. Write a `BPETokenizer` class with `encode(text) -> list[int]` and `decode(ids) -> str` that
(a) reserves id 0 for an `<unk>` token and uses it for any symbol outside the vocabulary,
(b) returns an empty list / empty string for empty input, (c) re-attaches punctuation to the preceding
word when decoding, and (d) round-trips any lower-case text whose characters are all in the alphabet.
Test all four behaviours, and observe what happens to the string `"Zürich 2026"`.

**Solution to Exercise 1**

The encoder reuses `bpe_encode_word` with the learned merge ranks and looks every token up in a
`stoi` dictionary with default 0; the decoder concatenates tokens, turns `</w>` into spaces and then
removes the space before punctuation with a regular expression. Round-tripping is exact for
in-alphabet text up to the normalizations we chose deliberately (lower-casing, collapsing whitespace).

In [ ]:
class BPETokenizer:
    """Ids for our from-scratch BPE: 0 = <unk>, then the learned vocabulary in sorted order."""

    def __init__(self, merges, vocab):
        self.ranks = {m: i for i, m in enumerate(merges)}
        self.itos = ["<unk>"] + sorted(vocab)
        self.stoi = {t: i for i, t in enumerate(self.itos)}

    def encode(self, text):
        ids = []
        for word in re.findall(r"[a-z']+|[^\sa-z']", text.lower()):
            ids.extend(self.stoi.get(tok, 0) for tok in bpe_encode_word(word, self.ranks))
        return ids

    def decode(self, ids):
        text = "".join(self.itos[i] for i in ids).replace(END, " ").strip()
        return re.sub(r" ([^\w\s'<])", r"\1", text)         # no space before punctuation (but keep it before <unk>)


bpe_tok = BPETokenizer(merges, bpe_vocab)
assert bpe_tok.encode("") == [] and bpe_tok.decode([]) == ""                    # (b) empty input
s = "hello, world! the king's men."
assert bpe_tok.decode(bpe_tok.encode(s)) == s, bpe_tok.decode(bpe_tok.encode(s))   # (c)+(d) round trip
print("round trip ok:", repr(bpe_tok.decode(bpe_tok.encode(s))))
ids = bpe_tok.encode("Zürich 2026")
print("Zürich 2026 ->", ids, "->", repr(bpe_tok.decode(ids)))                    # (a) unknown symbols
print("whitespace is normalized:", repr(bpe_tok.decode(bpe_tok.encode("two\n\n   words"))))
print("vocabulary size incl. <unk>:", len(bpe_tok.itos))

The accented `ü` and the digits were never seen during training, so they become `<unk>` and the
information is lost, which is exactly the failure that motivates *byte-level* BPE: with the 256 byte
values as the base alphabet there is nothing outside the vocabulary, and unusual characters merely cost
a few extra tokens. The whitespace example shows a second, deliberate loss: our pre-tokenizer discards
the layout of the text, which is acceptable for classification and unacceptable for code models.

### Exercise 2: Compare tokenizers on multilingual text

Take one sentence each in English, German, French, Russian, Japanese, plus a line of Python code and a
string with emoji, and count for each of the available tokenizers (our BPE, DistilBERT's WordPiece,
DistilGPT-2's byte-level BPE) how many tokens the string costs per character. Which tokenizer is most
even across languages? Which one loses information, and where?

**Solution to Exercise 2**

The table below prints tokens per character; `[UNK]` counts are reported for WordPiece since that is
where BERT's uncased vocabulary discards characters. Our own BPE (trained on English only, with an
alphabet of 26 letters) is included for comparison even though it is not a serious contender.

In [ ]:
multilingual = {"English": "The committee will reconvene tomorrow morning.",
                "German": "Der Ausschuss wird morgen früh wieder zusammentreten.",
                "French": "Le comité se réunira à nouveau demain matin.",
                "Russian": "Комитет соберётся снова завтра утром.",
                "Japanese": "委員会は明日の朝に再び開かれます。",
                "Python": "for i in range(10):\n    total += values[i] ** 2",
                "Emoji": "Great job 🎉🎉 see you at 5 🕔"}
tokenizers = {"our BPE": lambda s: bpe_encode(s)}
if bert_tok is not None:
    tokenizers["WordPiece (BERT)"] = bert_tok.tokenize
if gpt2_tok is not None:
    tokenizers["byte-BPE (GPT-2)"] = gpt2_tok.tokenize
header = f"{'language':10s} {'chars':>5s} " + " ".join(f"{n:>16s}" for n in tokenizers) + "   [UNK]"
print(header)
for lang, s in multilingual.items():
    counts = {n: f(s) for n, f in tokenizers.items()}
    row = f"{lang:10s} {len(s):5d} " + " ".join(f"{len(t):6d} ({len(t) / len(s):.2f}/ch)" for t in counts.values())
    unk = counts.get("WordPiece (BERT)", []).count("[UNK]")
    print(row + f"   {unk}")

GPT-2's byte-level BPE is the only tokenizer that never loses information, but it pays for it: Russian
and Japanese cost several times more tokens per character than English because its merges were learned
on English web text, so each Cyrillic or CJK character is split into two or three byte tokens. WordPiece
has a similar English bias and additionally maps some characters to `[UNK]`, which is unrecoverable.
Multilingual models (mBERT, XLM-R, Llama 3, Qwen) address this with vocabularies of 100k–250k pieces
learned on balanced multilingual data. Our toy BPE, with an alphabet of 26 letters, fails on everything
else, which is the correct behaviour for a tokenizer that was never meant to see other scripts.

### Exercise 3: Early stopping for the fine-tuning loop

Extend `fit` from Section 4.3 into `fit_early_stopping(...)` that monitors validation accuracy after
each epoch, stops when it has not improved for `patience` epochs, and restores the weights of the best
epoch before returning. Demonstrate it on a model that trains in seconds. A convenient choice that runs
even without pretrained weights is a bag-of-embeddings classifier that follows the Hugging Face
calling convention (accepts `input_ids`, `attention_mask`, `labels`; returns an object with `.loss`
and `.logits`), so that `train_one_epoch`, `evaluate` and the DataLoaders of Section 4 work unchanged.

**Solution to Exercise 3**

The key detail is to keep a *copy* of the best `state_dict` (moved to the CPU and cloned; a plain
reference would be overwritten by the next optimizer step) and to load it back at the end. The same
function works for DistilBERT, LoRA models and peft models, because it only relies on the
`model(**batch)` interface.

In [ ]:
class TinyBagClassifier(nn.Module):
    """Bag-of-embeddings classifier with the Hugging Face call/return conventions."""

    def __init__(self, vocab_size, dim=64, num_labels=2, pad_id=0):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim, padding_idx=pad_id)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(dim, num_labels))

    def forward(self, input_ids, attention_mask, labels=None, **unused):     # ignore e.g. token_type_ids
        m = attention_mask.unsqueeze(-1).float()
        pooled = (self.emb(input_ids) * m).sum(1) / m.sum(1).clamp(min=1.0)
        logits = self.head(pooled)
        loss = F.cross_entropy(logits, labels) if labels is not None else None
        return SimpleNamespace(loss=loss, logits=logits)


def fit_early_stopping(model, train_loader, val_loader, epochs, lr, patience=2, monitor="acc",
                       weight_decay=0.01, warmup_frac=0.1, tag=""):
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    total_steps = epochs * len(train_loader)
    scheduler = linear_warmup_decay(optimizer, int(warmup_frac * total_steps), total_steps)
    best_score, best_state, best_epoch, bad_epochs, history = -float("inf"), None, 0, 0, []
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
        val = evaluate(model, val_loader)
        history.append({"epoch": epoch, "train_loss": train_loss, **val})
        improved = val[monitor] > best_score
        print(f"{tag}epoch {epoch:2d} | train loss {train_loss:.4f} | val loss {val['loss']:.4f} | "
              f"val acc {val['acc']:.3f} | {time.time() - t0:.1f}s" + ("  *" if improved else ""))
        if improved:
            best_score, best_epoch, bad_epochs = val[monitor], epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                print(f"{tag}early stopping: no improvement in {patience} epochs")
                break
    model.load_state_dict(best_state)
    print(f"{tag}restored epoch {best_epoch} (val {monitor} {best_score:.3f})")
    return history


torch.manual_seed(SEED)
pad_id = getattr(cls_tok, "pad_token_id", 0)
tiny = TinyBagClassifier(len(cls_tok), pad_id=pad_id).to(device)
tiny_hist = fit_early_stopping(tiny, train_loader, val_loader, epochs=10 if FAST_RUN else 20,
                               lr=5e-3, patience=3, tag="[tiny] ")
final = evaluate(tiny, val_loader)
print(f"after restoring the best weights: val acc {final['acc']:.3f}")

The bag-of-embeddings model overfits quickly (its validation accuracy peaks after a few epochs while
the training loss keeps falling), which makes the early-stopping trigger visible within seconds. The
final evaluation confirms that the restored weights reproduce the best epoch's accuracy exactly. With
DistilBERT the same function typically stops after two to three epochs on SST-2 subsets; compare
`results["full fine-tuning"]` with what you get from `fit_early_stopping(ft_model, ...)` on a GPU.

### Exercise 4: LoRA for the feed-forward layers too

Section 5 adapted only the query and value projections. Extend the adaptation to all six linear
layers of each Transformer block (`q_lin`, `k_lin`, `v_lin`, `out_lin`, `ffn.lin1`, `ffn.lin2`).
First predict the number of LoRA parameters per block from the formula $r(d_{\text{in}} +
d_{\text{out}})$ and verify it in code; then fine-tune (same budget as Section 5) and compare
accuracy and trainable-parameter count with the attention-only run.

**Solution to Exercise 4**

Per block with $r = 8$: four $768 \times 768$ projections contribute $4 \times 8 \times 1536 =
49{,}152$ parameters and the two feed-forward layers ($768 \to 3072$ and $3072 \to 768$) contribute
$2 \times 8 \times 3840 = 61{,}440$, so $110{,}592$ per block and $663{,}552$ for six blocks, about
4.5 times the attention-only adapter. The parameter check below runs on a small stand-in module even
without the pretrained weights; the fine-tuning run needs them.

In [ ]:
def lora_param_count(model):
    return sum(m.lora_A.numel() + m.lora_B.numel() for m in lora_modules(model))


# parameter formula check on a stand-in block with the DistilBERT shapes
stand_in = nn.ModuleDict({"q_lin": nn.Linear(768, 768), "k_lin": nn.Linear(768, 768),
                          "v_lin": nn.Linear(768, 768), "out_lin": nn.Linear(768, 768),
                          "lin1": nn.Linear(768, 3072), "lin2": nn.Linear(3072, 768)})
ALL_TARGETS = ("q_lin", "k_lin", "v_lin", "out_lin", "lin1", "lin2")
apply_lora(stand_in, targets=ALL_TARGETS, r=LORA_R, alpha=LORA_ALPHA)
predicted = 4 * LORA_R * (768 + 768) + 2 * LORA_R * (768 + 3072)
print(f"LoRA parameters per block: counted {lora_param_count(stand_in):,} | predicted {predicted:,}")

lora_all = hf_load(AutoModelForSequenceClassification, "distilbert-base-uncased", num_labels=2)
if lora_all is not None:
    n = apply_lora(lora_all, targets=ALL_TARGETS, r=LORA_R, alpha=LORA_ALPHA, dropout=0.05)
    mark_only_lora_trainable(lora_all, also_train=("pre_classifier", "classifier"))
    lora_all = lora_all.to(device)
    attn_only = lora_param_count(lora_model) if lora_model is not None else 0
    print(f"replaced {n} layers; LoRA parameters {lora_param_count(lora_all):,} "
          f"(attention-only run had {attn_only:,})")
    torch.manual_seed(SEED)
    hist = fit(lora_all, train_loader, val_loader, epochs=EPOCHS_FT, lr=LR_LORA, tag="[LoRA all] ")
    results["LoRA all linears"] = {"trainable": count_params(lora_all)[1], **hist[-1]}
    for name in ["LoRA from scratch", "LoRA all linears"]:
        if name not in results:
            continue
        r = results[name]
        print(f"{name:20s} trainable {r['trainable']:9,d}  val acc {r['acc']:.3f}  F1 {r['f1']:.3f}")
    lora_all = None
    gc.collect()
else:
    skip("the fine-tuning part of this exercise needs the DistilBERT weights")

Adapting every linear layer usually gives a small improvement for a modest increase in adapter size,
and it is the default recommendation of the QLoRA paper ("apply LoRA to all linear layers"). On a
budget this small the difference is within run-to-run noise, which is itself a lesson: compare methods
over several seeds before believing a half-point gap (Notebook 17 · Capstone Projects makes this a
rule).

### Exercise 5 (challenge): DoRA from scratch, compared with LoRA

**DoRA** (Weight-Decomposed Low-Rank Adaptation) writes a weight matrix as a magnitude times a
direction, $W = m \odot \frac{V}{\|V\|}$, where the norm is taken per output unit (per row of the
PyTorch weight), applies the LoRA update to the direction only, and trains the magnitude vector $m$
separately:

$$
W' = m \odot \frac{W + \frac{\alpha}{r} B A}{\big\|W + \frac{\alpha}{r} B A\big\|_{\text{row}}},
\qquad m \in \mathbb{R}^{d_{\text{out}}} \text{ initialized to } \|W\|_{\text{row}}.
$$

Implement `DoRALinear` with the same interface as `LoRALinear`, verify that it equals the base layer
at initialization, and compare LoRA and DoRA at equal rank on a synthetic adaptation task: start from
a random "pretrained" linear map $W_0$ and learn to imitate a target map $W^* = D\,(W_0 + \text{low-rank})$
whose rows have been rescaled by a diagonal $D$, using the mean-squared error between outputs on random
inputs. Which method fits better, and why? (Alternatively implement IA3, which learns only per-channel
scaling vectors.)

**Solution to Exercise 5**

The magnitude vector gives DoRA a cheap way to express exactly the row-rescaling that the target
contains, while LoRA must spend rank on it. The experiment is deliberately constructed to show this,
and it is set up in the *fine-tuning regime*: the low-rank change is small compared with the
pretrained weights themselves, as it is in practice. On real fine-tuning tasks the DoRA paper reports
consistent gains of one to a few points over LoRA at essentially the same parameter count, and the
same code applies to DistilBERT through `apply_lora`-style replacement.

In [ ]:
class DoRALinear(nn.Module):
    """Weight-decomposed low-rank adaptation of a frozen nn.Linear (Liu et al., 2024)."""

    def __init__(self, base: nn.Linear, r=8, alpha=16.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.scaling = alpha / r
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        self.magnitude = nn.Parameter(base.weight.detach().norm(dim=1).clone())     # (out,)

    def adapted_weight(self):
        V = self.base.weight + self.scaling * (self.lora_B @ self.lora_A)           # direction (out, in)
        return V * (self.magnitude / V.norm(dim=1)).unsqueeze(1)

    def forward(self, x):
        return F.linear(x, self.adapted_weight(), self.base.bias)


torch.manual_seed(SEED)
d_in, d_out, rank = 64, 32, 4
W0 = nn.Linear(d_in, d_out)
x_test = torch.randn(8, d_in)
print("DoRA == base at init:", torch.allclose(DoRALinear(W0, r=rank)(x_test), W0(x_test), atol=1e-6))

# synthetic target: rows of (W0 + small low-rank change) rescaled by a diagonal D
LOW_RANK_SCALE = 0.05                      # entries of W0 are ~0.07: the change is modest, as in fine-tuning
with torch.no_grad():
    low_rank = LOW_RANK_SCALE * torch.randn(d_out, rank) @ torch.randn(rank, d_in) / math.sqrt(rank)
    D = torch.exp(0.5 * torch.randn(d_out, 1))
    W_target = D * (W0.weight + low_rank)
    b_target = W0.bias.clone()


def adapt(adapter_cls, steps=300, lr=1e-2):
    torch.manual_seed(SEED)
    base = nn.Linear(d_in, d_out)
    base.load_state_dict(W0.state_dict())
    layer = adapter_cls(base, r=rank, alpha=2 * rank)
    opt = torch.optim.Adam([p for p in layer.parameters() if p.requires_grad], lr=lr)
    losses = []
    for _ in range(steps):
        xb = torch.randn(256, d_in)
        target = xb @ W_target.t() + b_target
        loss = F.mse_loss(layer(xb), target)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    n_train = sum(p.numel() for p in layer.parameters() if p.requires_grad)
    return losses, n_train


lora_losses, n_lora = adapt(LoRALinear)
dora_losses, n_dora = adapt(DoRALinear)
plt.semilogy(lora_losses, label=f"LoRA r={rank} ({n_lora} params)")
plt.semilogy(dora_losses, label=f"DoRA r={rank} ({n_dora} params)")
plt.xlabel("step"); plt.ylabel("MSE to target map"); plt.title("Adapting a linear map: LoRA vs DoRA")
plt.legend(); plt.tight_layout(); plt.show()
print(f"final MSE: LoRA {lora_losses[-1]:.4f} | DoRA {dora_losses[-1]:.4f}")

DoRA drives the error far below LoRA's plateau with only $d_{\text{out}} = 32$ extra parameters (the
target is exactly representable, so with enough steps the error goes to zero), while LoRA plateaus: a
per-row rescaling is a full-rank diagonal change that no rank-4 update can represent exactly, whereas
DoRA's magnitude vector absorbs it and leaves the low-rank part free to model the remaining direction
change. IA3 (Liu et al., 2022) takes this idea to its extreme: it learns *only* scaling vectors (on
keys, values and the feed-forward hidden activations) and no low-rank matrices at all, which is enough
for surprisingly many tasks at roughly a tenth of LoRA's parameter count.

> **Try it.** Set `LOW_RANK_SCALE = 0.3`, so that the change is several times larger than the
> pretrained weights themselves, and re-run. Which method is ahead after 300 steps now, and by how much
> has DoRA's advantage shrunk? The rank-4 part of the target is now the dominant term, which LoRA fits
> directly, whereas DoRA parametrizes the *direction* of each weight row and has to turn it almost
> completely away from $W_0$ through a normalization, a slower and worse-conditioned path. The
> decomposition pays off precisely when fine-tuning is a modest correction of a good pretrained model,
> which is the regime it was designed for.

## Summary

- Subword tokenization (BPE, WordPiece, Unigram) learns its vocabulary from data, BPE and WordPiece by
  merging frequent symbol pairs and Unigram by pruning a large candidate set; the number of merges sets
  BPE's vocabulary size, and byte-level variants never produce unknown tokens. Always use the tokenizer
  that belongs to the pretrained model.
- Hugging Face tokenizers add special tokens, produce attention masks, pad and truncate batches and
  return offsets; dynamic padding (pad to the longest sequence in the batch) saves most of the compute
  wasted on padding.
- word2vec's skip-gram with negative sampling learns static embeddings from co-occurrence by binary
  classification of (center, context) pairs against sampled negatives; cosine similarity of the
  resulting vectors captures word similarity, and contextual embeddings from pretrained Transformers
  generalize the idea to one vector per occurrence.
- `AutoModel` and its task-specific variants load pretrained checkpoints into ordinary `nn.Module`s;
  `generate` implements the decoding strategies, `pipeline` bundles everything for quick inference.
- Fine-tuning a pretrained encoder for classification needs AdamW with a small learning rate, linear
  warmup and decay, gradient clipping and a few epochs; a hand-written loop and the `Trainer` API
  implement the same recipe and should agree.
- LoRA adds a trainable low-rank update $\frac{\alpha}{r} B A$ to frozen linear layers, starting from
  $B = 0$; it trains about 1% of the parameters, matches full fine-tuning closely, and can be merged
  into the weights for free inference. `peft` implements it (and adapters, prefix tuning, IA3, QLoRA,
  DoRA) for any `transformers` model.
- Modern LLMs are trained in stages: next-token pretraining, supervised fine-tuning with the loss masked
  to response tokens, and preference optimization; DPO turns the KL-regularized RLHF objective into a
  logistic loss on the policy's log-ratios against a reference model, with $\beta$ controlling how far
  the policy may move.
- Attention memory is quadratic in sequence length; truncate sensibly, use mixed precision, and read
  the model card and license before you build on a checkpoint. Treat benchmark numbers with suspicion
  until contamination is ruled out.

## Further reading

- Sennrich, Haddow and Birch, 2016, *Neural Machine Translation of Rare Words with Subword Units* (Byte-Pair Encoding for NLP).
- Kudo and Richardson, 2018, *SentencePiece: A simple and language independent subword tokenizer and detokenizer for Neural Text Processing*.
- Mikolov, Chen, Corrado and Dean, 2013, *Efficient Estimation of Word Representations in Vector Space*; Mikolov, Sutskever, Chen, Corrado and Dean, 2013, *Distributed Representations of Words and Phrases and their Compositionality* (negative sampling).
- Devlin, Chang, Lee and Toutanova, 2019, *BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding*; Sanh, Debut, Chaumond and Wolf, 2019, *DistilBERT, a distilled version of BERT*.
- Radford et al., 2019, *Language Models are Unsupervised Multitask Learners* (GPT-2 and byte-level BPE).
- Wolf et al., 2020, *Transformers: State-of-the-Art Natural Language Processing*; documentation at https://huggingface.co/docs/transformers and https://huggingface.co/docs/peft.
- Hu et al., 2021, *LoRA: Low-Rank Adaptation of Large Language Models*; Dettmers, Pagnoni, Holtzman and Zettlemoyer, 2023, *QLoRA: Efficient Finetuning of Quantized LLMs*; Liu et al., 2024, *DoRA: Weight-Decomposed Low-Rank Adaptation*.
- Ouyang et al., 2022, *Training language models to follow instructions with human feedback* (InstructGPT/RLHF); Rafailov et al., 2023, *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*.
- Mitchell et al., 2019, *Model Cards for Model Reporting*.
- Jurafsky and Martin, *Speech and Language Processing* (3rd edition draft), chapters on subword tokenization, vector semantics and fine-tuning.